<a href="https://colab.research.google.com/github/MartinS2804/bangkok-property-guide-ai/blob/main/Bangkok_Foreign_Investor_AI_Chatbot_V4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# CELL 1 - Imports & Global Configuration

import re
import math
import json
import statistics
from typing import List, Dict, Optional, Tuple

import gradio as gr

from IPython.display import display, Markdown


APP_NAME = "TH Bangkok Foreign Investor AI"
APP_VERSION = "V4"

LEGAL_DISCLAIMER = (
    "This chatbot is an educational investment-screening prototype and does not "
    "provide legal, tax, engineering, valuation or financial advice. Regulations, "
    "market conditions and transaction requirements may change. Users should verify "
    "current requirements and property-specific information with competent authorities, "
    "qualified legal advisers, banks, engineers, valuers and other professional advisers."
)

DATA_LIMITATION_NOTE = (
    "Area prices, yields, flood, soil and market indicators are screening-level inputs. "
    "They are not transaction-level appraisal evidence and should not replace "
    "property-specific legal, technical and financial due diligence."
)


print(f"✓ {APP_NAME} {APP_VERSION} base environment loaded successfully")
print("✓ Imports loaded")
print("✓ Global disclaimer loaded")

✓ TH Bangkok Foreign Investor AI V4 base environment loaded successfully
✓ Imports loaded
✓ Global disclaimer loaded


In [2]:
# CELL 2 - Structured Bangkok Knowledge Base

BANGKOK_ZONES = {

    "Sukhumvit Core": {
        "aliases": ["sukhumvit core", "asok", "nana", "phrom phong"],
        "midpoint_price_sqm": 190000,
        "gross_yield": 4.75,
        "transit_score": 10,
        "growth_score": 7,
        "international_demand_score": 10,
        "transit": "BTS Sukhumvit Line / MRT interchange access",
        "property_focus": "Premium condominium",
        "neighborhood": "Prime central Bangkok with strong international tenant and expatriate demand",
        "investor_profile": "Premium / international rental demand",
        "flood": "Project- and street-level drainage should still be reviewed despite central location",
        "soil": "Bangkok soft-clay conditions; project-specific foundation review recommended"
    },

    "Thonglor - Ekkamai": {
        "aliases": ["thonglor", "thong lo", "ekkamai", "ekkami"],
        "midpoint_price_sqm": 185000,
        "gross_yield": 4.75,
        "transit_score": 9,
        "growth_score": 7,
        "international_demand_score": 10,
        "transit": "BTS Sukhumvit Line",
        "property_focus": "Premium condominium / lifestyle residential",
        "neighborhood": "High-end lifestyle district with strong expatriate and affluent tenant demand",
        "investor_profile": "Premium / lifestyle / international tenants",
        "flood": "Check soi-level drainage and vehicle access during heavy rainfall",
        "soil": "Bangkok soft-clay conditions; building-specific engineering review required"
    },

    "Silom - Sathorn": {
        "aliases": ["silom", "sathorn", "sathon"],
        "midpoint_price_sqm": 230000,
        "gross_yield": 4.25,
        "transit_score": 10,
        "growth_score": 6,
        "international_demand_score": 9,
        "transit": "BTS Silom Line / MRT",
        "property_focus": "Premium condominium / CBD residential",
        "neighborhood": "Major business district with professional and international tenant demand",
        "investor_profile": "CBD / premium / defensive rental demand",
        "flood": "Street-level drainage and access should be checked for specific projects",
        "soil": "Soft-clay Bangkok geology; high-rise foundation systems require project-specific review"
    },

    "Central Lumpini": {
        "aliases": ["central lumpini", "lumpini", "lumphini", "wireless road", "ploenchit", "phloen chit"],
        "midpoint_price_sqm": 335000,
        "gross_yield": 3.75,
        "transit_score": 10,
        "growth_score": 7,
        "international_demand_score": 10,
        "transit": "BTS / MRT central access",
        "property_focus": "Luxury condominium",
        "neighborhood": "Ultra-prime central district with embassies, offices and luxury residential demand",
        "investor_profile": "Luxury / capital preservation / international demand",
        "flood": "Project access and drainage remain relevant even in prime central areas",
        "soil": "Bangkok soft-clay conditions; rely on project engineering and foundation documentation"
    },

    "Rama 9 - Ratchada": {
        "aliases": ["rama 9", "rama ix", "ratchada", "ratchadaphisek"],
        "midpoint_price_sqm": 140000,
        "gross_yield": 5.50,
        "transit_score": 9,
        "growth_score": 9,
        "international_demand_score": 8,
        "transit": "MRT / major road connectivity",
        "property_focus": "Condominium / mixed urban residential",
        "neighborhood": "Emerging business and residential district with strong growth narrative",
        "investor_profile": "Growth / rental income",
        "flood": "Check drainage, road access and project elevation at local level",
        "soil": "Soft-clay conditions require project-specific foundation and engineering review"
    },

    "Huai Khwang": {
        "aliases": ["huai khwang", "huay khwang"],
        "midpoint_price_sqm": 117500,
        "gross_yield": 5.75,
        "transit_score": 9,
        "growth_score": 8,
        "international_demand_score": 7,
        "transit": "MRT Blue Line",
        "property_focus": "Condominium / rental residential",
        "neighborhood": "Dense residential and commercial area with local and international tenant demand",
        "investor_profile": "Rental income / value",
        "flood": "Street-level drainage and low-lying access should be reviewed",
        "soil": "Bangkok soft-clay conditions; project engineering review recommended"
    },

    "Ari - Phaya Thai": {
        "aliases": ["ari", "phaya thai", "phayathai"],
        "midpoint_price_sqm": 160000,
        "gross_yield": 5.00,
        "transit_score": 9,
        "growth_score": 8,
        "international_demand_score": 8,
        "transit": "BTS / Airport Rail Link connectivity",
        "property_focus": "Condominium / urban residential",
        "neighborhood": "Established residential district with cafés, offices and strong urban lifestyle appeal",
        "investor_profile": "Balanced / lifestyle / professional tenants",
        "flood": "Check local drainage and building access during heavy rainfall",
        "soil": "Soft-clay Bangkok geology; site-specific engineering due diligence recommended"
    },

    "Ratchathewi - Victory Monument": {
        "aliases": ["ratchathewi", "victory monument", "victory"],
        "midpoint_price_sqm": 160000,
        "gross_yield": 5.15,
        "transit_score": 10,
        "growth_score": 7,
        "international_demand_score": 8,
        "transit": "BTS / Airport Rail Link / major bus connectivity",
        "property_focus": "Condominium / central residential",
        "neighborhood": "Highly connected central district near universities, hospitals and employment centers",
        "investor_profile": "Transit / central rental demand",
        "flood": "Urban drainage and access should be checked at project level",
        "soil": "Bangkok soft-clay conditions; project foundation documentation required"
    },

    "On Nut - Phra Khanong": {
        "aliases": ["on nut", "onnut", "phra khanong", "phrakanong"],
        "midpoint_price_sqm": 115000,
        "gross_yield": 6.05,
        "transit_score": 9,
        "growth_score": 8,
        "international_demand_score": 8,
        "transit": "BTS Sukhumvit Line",
        "property_focus": "Condominium",
        "neighborhood": "Value-oriented residential area with growing international tenant base",
        "investor_profile": "Yield / value / first-time investor",
        "flood": "Check soi-level drainage and access during heavy rainfall",
        "soil": "Bangkok soft-clay conditions"
    },

    "Punnawithi - Udom Suk": {
        "aliases": ["punnawithi", "udom suk", "udomsuk"],
        "midpoint_price_sqm": 100000,
        "gross_yield": 6.05,
        "transit_score": 8,
        "growth_score": 9,
        "international_demand_score": 7,
        "transit": "BTS Sukhumvit Line",
        "property_focus": "Condominium / value residential",
        "neighborhood": "Affordable eastern corridor benefiting from transit and employment decentralization",
        "investor_profile": "Growth / affordability",
        "flood": "Check local drainage, soi access and heavy-rain conditions",
        "soil": "Soft-clay conditions require normal Bangkok engineering due diligence"
    },

    "Bang Na": {
        "aliases": ["bang na", "bangna"],
        "midpoint_price_sqm": 90000,
        "gross_yield": 6.00,
        "transit_score": 7,
        "growth_score": 9,
        "international_demand_score": 6,
        "transit": "BTS / major road / eastern corridor connections",
        "property_focus": "Condominium / suburban residential",
        "neighborhood": "Growth corridor linked to eastern Bangkok employment, retail and infrastructure",
        "investor_profile": "Growth / affordability",
        "flood": "Drainage and low-lying site conditions require local verification",
        "soil": "Soft soil and low-lying conditions require site-specific engineering review"
    },

    "Bang Sue - Tao Poon": {
        "aliases": ["bang sue", "bangsue", "tao poon", "taopoon"],
        "midpoint_price_sqm": 102500,
        "gross_yield": 6.25,
        "transit_score": 10,
        "growth_score": 8,
        "international_demand_score": 6,
        "transit": "MRT / rail hub connectivity",
        "property_focus": "Condominium / transit-oriented residential",
        "neighborhood": "Transit-led northern Bangkok growth zone with major rail infrastructure",
        "investor_profile": "Transit-led growth / value",
        "flood": "Check road drainage and project access at local level",
        "soil": "Bangkok soft-clay conditions; engineering due diligence required"
    },

    "Chatuchak - Ratchayothin": {
        "aliases": ["chatuchak", "ratchayothin", "ratchayothin"],
        "midpoint_price_sqm": 125000,
        "gross_yield": 5.40,
        "transit_score": 9,
        "growth_score": 8,
        "international_demand_score": 7,
        "transit": "BTS / MRT connectivity",
        "property_focus": "Condominium / urban residential",
        "neighborhood": "Large employment, retail and residential district with strong connectivity",
        "investor_profile": "Balanced / transit / rental demand",
        "flood": "Local drainage and road access should be checked",
        "soil": "Soft-clay Bangkok geology; project-specific foundation review recommended"
    },

    "Lat Phrao": {
        "aliases": ["lat phrao", "lad phrao", "ladprao", "latprao"],
        "midpoint_price_sqm": 92500,
        "gross_yield": 5.75,
        "transit_score": 8,
        "growth_score": 8,
        "international_demand_score": 5,
        "transit": "MRT / rail expansion / major road network",
        "property_focus": "Condominium / local residential",
        "neighborhood": "Large local residential catchment with improving transit connectivity",
        "investor_profile": "Value / local rental demand",
        "flood": "Flood and drainage checks are important at street and soi level",
        "soil": "Soft clay and low-lying conditions require engineering review"
    },

    "Ramkhamhaeng - Bang Kapi": {
        "aliases": ["ramkhamhaeng", "bang kapi", "bangkapi", "hua mak", "huamark"],
        "midpoint_price_sqm": 82500,
        "price_range": (55000, 110000),
        "gross_yield": 6.15,
        "yield_range": (5.3, 7.0),
        "transit_score": 7,
        "growth_score": 8,
        "international_demand_score": 5,
        "transit": "Airport Rail Link / rail expansion / major road connections",
        "property_focus": "Condominium / Residential",
        "neighborhood": "Universities, large residential population and local retail demand",
        "investor_profile": "Budget / local rental demand",
        "flood": "Flood and drainage due diligence is particularly important at street level",
        "soil": "Soft clay; low-lying site conditions require engineering review"
    },

    "Riverside - Charoen Krung": {
        "aliases": ["riverside", "charoen krung"],
        "midpoint_price_sqm": 250000,
        "gross_yield": 4.25,
        "transit_score": 7,
        "growth_score": 7,
        "international_demand_score": 9,
        "transit": "River transport / BTS connections depending on project",
        "property_focus": "Luxury condominium / riverside residential",
        "neighborhood": "Premium riverfront corridor with hotel, tourism and luxury residential demand",
        "investor_profile": "Luxury / lifestyle / international buyers",
        "flood": "River proximity makes project elevation, flood protection and access important",
        "soil": "River-adjacent soft soil requires project-specific foundation and geotechnical review"
    },

    "Charoen Nakhon - Khlong San": {
        "aliases": ["charoen nakhon", "charoen nakorn", "khlong san", "klong san"],
        "midpoint_price_sqm": 160000,
        "gross_yield": 5.25,
        "transit_score": 8,
        "growth_score": 8,
        "international_demand_score": 8,
        "transit": "Gold Line / river crossings / BTS connectivity",
        "property_focus": "Condominium / riverside residential",
        "neighborhood": "Rapidly upgraded west-bank district with major retail and residential investment",
        "investor_profile": "Growth / riverside / international demand",
        "flood": "River proximity and local drainage require project-level review",
        "soil": "Soft river-adjacent soils require engineering and foundation due diligence"
    },

    "Rama III - Yannawa": {
        "aliases": ["rama iii", "rama 3", "yannawa", "yan nawa"],
        "midpoint_price_sqm": 105000,
        "gross_yield": 5.75,
        "transit_score": 7,
        "growth_score": 7,
        "international_demand_score": 6,
        "transit": "BRT / major roads / river corridor",
        "property_focus": "Condominium / residential",
        "neighborhood": "Established riverside-adjacent residential and business corridor",
        "investor_profile": "Value / income / long-term residential",
        "flood": "Check river-related flood exposure, drainage and access",
        "soil": "Soft clay and river-adjacent conditions require engineering review"
    },

    "Pinklao - Bang Phlat": {
        "aliases": ["pinklao", "pin klao", "bang phlat", "bangphlat"],
        "midpoint_price_sqm": 92500,
        "gross_yield": 5.75,
        "transit_score": 7,
        "growth_score": 7,
        "international_demand_score": 5,
        "transit": "MRT / road and river connections",
        "property_focus": "Condominium / local residential",
        "neighborhood": "West Bangkok residential market with improving rail access",
        "investor_profile": "Value / local demand",
        "flood": "Low-lying areas require careful drainage and flood-history checks",
        "soil": "Soft clay and river-influenced ground conditions require engineering review"
    },

    "Talat Phlu - Wutthakat": {
        "aliases": ["talat phlu", "talad phlu", "wutthakat", "wutthakat"],
        "midpoint_price_sqm": 77500,
        "gross_yield": 6.35,
        "transit_score": 8,
        "growth_score": 7,
        "international_demand_score": 5,
        "transit": "BTS Silom Line",
        "property_focus": "Affordable condominium / residential",
        "neighborhood": "Affordable west Bangkok residential corridor with BTS access",
        "investor_profile": "Yield / affordability / local rental demand",
        "flood": "Street-level drainage and low-lying access should be checked carefully",
        "soil": "Bangkok soft clay; building-specific engineering review recommended"
    }
}


# ------------------------------------------------------------
# Build one normalized alias index for later NLP/entity detection
# ------------------------------------------------------------

ZONE_ALIAS_INDEX = {}

for canonical_zone, data in BANGKOK_ZONES.items():

    # canonical name itself
    ZONE_ALIAS_INDEX[canonical_zone.lower()] = canonical_zone

    # aliases
    for alias in data["aliases"]:
        ZONE_ALIAS_INDEX[alias.lower()] = canonical_zone


# ------------------------------------------------------------
# Basic database validation
# ------------------------------------------------------------

required_fields = [
    "aliases",
    "midpoint_price_sqm",
    "gross_yield",
    "transit_score",
    "growth_score",
    "international_demand_score",
    "transit",
    "property_focus",
    "neighborhood",
    "investor_profile",
    "flood",
    "soil"
]

database_errors = []

for zone_name, zone_data in BANGKOK_ZONES.items():
    for field in required_fields:
        if field not in zone_data:
            database_errors.append(
                f"{zone_name}: missing field '{field}'"
            )

if database_errors:
    print("⚠ Database validation problems:")
    for error in database_errors:
        print("-", error)
else:
    print("✓ Bangkok structured knowledge base loaded successfully")
    print(f"✓ {len(BANGKOK_ZONES)} investment zones loaded")
    print(f"✓ {len(ZONE_ALIAS_INDEX)} location names / aliases indexed")
    print("✓ Database structure validation passed")

✓ Bangkok structured knowledge base loaded successfully
✓ 20 investment zones loaded
✓ 85 location names / aliases indexed
✓ Database structure validation passed


In [3]:
# CELL 3 - Legal & Property Knowledge Base

PROPERTY_TYPES = {

    "condominium": {
        "aliases": [
            "condo",
            "condominium",
            "apartment unit",
            "residential unit"
        ],
        "category": "Residential",
        "foreign_investor_relevance": "High",
        "description": (
            "Individually owned unit in a registered condominium building."
        )
    },

    "house": {
        "aliases": [
            "house",
            "single house",
            "detached house",
            "townhouse",
            "townhome",
            "villa"
        ],
        "category": "Residential",
        "foreign_investor_relevance": "Medium",
        "description": (
            "Residential building where ownership of the building and ownership "
            "of the underlying land must be considered separately."
        )
    },

    "land": {
        "aliases": [
            "land",
            "plot",
            "land plot",
            "raw land",
            "vacant land"
        ],
        "category": "Land",
        "foreign_investor_relevance": "Restricted",
        "description": (
            "Land ownership is a major regulatory issue for foreign investors."
        )
    },

    "commercial": {
        "aliases": [
            "commercial",
            "office",
            "office building",
            "retail",
            "shop",
            "hotel",
            "hospitality"
        ],
        "category": "Commercial",
        "foreign_investor_relevance": "Case-specific",
        "description": (
            "Commercial property requiring transaction-specific ownership, "
            "zoning, licensing and investment review."
        )
    },

    "industrial": {
        "aliases": [
            "industrial",
            "warehouse",
            "factory",
            "manufacturing",
            "distribution center",
            "logistics"
        ],
        "category": "Industrial",
        "foreign_investor_relevance": "Case-specific",
        "description": (
            "Industrial real estate where land ownership, zoning, permits and "
            "business-use restrictions require specialist review."
        )
    },

    "mixed_use": {
        "aliases": [
            "mixed use",
            "mixed-use",
            "mixed use building",
            "mixed-use building"
        ],
        "category": "Mixed-use",
        "foreign_investor_relevance": "Case-specific",
        "description": (
            "Property combining multiple uses such as residential, retail "
            "and office space."
        )
    }
}


LEGAL_RULES = {

    "foreign_condominium_freehold": {
        "status": "GENERALLY PERMITTED",
        "rule": (
            "Foreign individuals may generally own qualifying condominium "
            "units in Thailand in their own name, subject to applicable "
            "statutory conditions and the foreign ownership quota."
        ),
        "key_condition": (
            "Foreign ownership must generally remain within the applicable "
            "condominium foreign-ownership quota."
        ),
        "due_diligence": [
            "Verify that the condominium is legally registered.",
            "Confirm the remaining foreign ownership quota with the condominium juristic person.",
            "Verify the unit title and ownership records with the competent Land Office.",
            "Confirm foreign-fund remittance and transfer-document requirements before completion.",
            "Review common-area fees, sinking fund obligations and building financial statements."
        ]
    },

    "foreign_land_freehold": {
        "status": "GENERALLY RESTRICTED",
        "rule": (
            "Foreign individuals generally cannot directly own land in Thailand "
            "under ordinary circumstances, subject to limited statutory exceptions."
        ),
        "key_condition": (
            "A foreign investor should not assume ordinary direct freehold land "
            "ownership is legally available."
        ),
        "due_diligence": [
            "Obtain qualified Thai legal advice before committing funds.",
            "Verify the land title at the competent Land Office.",
            "Confirm the exact permitted ownership or investment structure.",
            "Do not use nominee arrangements to circumvent Thai law.",
            "Review zoning, access, easements and development restrictions."
        ]
    },

    "foreign_house": {
        "status": "STRUCTURE-DEPENDENT",
        "rule": (
            "A foreign investor may face different rules for ownership of a "
            "building and ownership of the underlying land."
        ),
        "key_condition": (
            "The legal structure of the land and the building must be reviewed separately."
        ),
        "due_diligence": [
            "Clarify who legally owns the land.",
            "Clarify who legally owns the building.",
            "Review any lease agreement affecting the land.",
            "Verify building registration, permits and title-related documents.",
            "Obtain transaction-specific Thai legal advice."
        ]
    },

    "leasehold": {
        "status": "POTENTIALLY AVAILABLE",
        "rule": (
            "Leasehold may provide contractual use rights without transferring "
            "freehold ownership of the underlying land."
        ),
        "key_condition": (
            "Lease term, registration, renewal language, transfer rights and "
            "termination provisions require contract-specific legal review."
        ),
        "due_diligence": [
            "Verify the legal owner and title of the leased property.",
            "Check whether the lease must be registered.",
            "Review lease term and renewal clauses carefully.",
            "Review assignment, inheritance and termination provisions.",
            "Do not treat contractual renewal expectations as guaranteed ownership rights."
        ]
    }
}


THAI_LAND_TITLES = {

    "chanote": {
        "formal_name": "Chanote (N.S.4)",
        "ownership_quality": "Full title deed",
        "boundary_risk": "Very low relative risk",
        "survey_basis": "Formal surveyed boundaries / boundary markers",
        "transferability": "Generally transferable, subject to applicable law",
        "screening_view": (
            "Usually the strongest land-title form for ownership verification, "
            "but the specific deed and Land Office records must still be checked."
        )
    },

    "nor_sor_3_khor": {
        "formal_name": "Nor Sor 3 Khor (N.S.3K)",
        "ownership_quality": "Confirmed claim / use certificate",
        "boundary_risk": "Moderate relative risk",
        "survey_basis": "Mapped / surveyed with less certainty than Chanote",
        "transferability": "May be transferable subject to legal requirements",
        "screening_view": (
            "Requires more careful title, boundary and Land Office verification "
            "than a Chanote title."
        )
    },

    "nor_sor_3": {
        "formal_name": "Nor Sor 3 (N.S.3)",
        "ownership_quality": "Confirmed claim / use certificate",
        "boundary_risk": "Higher relative risk",
        "survey_basis": "Less precise boundary definition",
        "transferability": "May be transferable subject to legal requirements",
        "screening_view": (
            "Boundary and legal due diligence is particularly important."
        )
    },

    "sor_por_gor": {
        "formal_name": "Sor Por Gor",
        "ownership_quality": "Agricultural-use right / restricted land document",
        "boundary_risk": "Very high for ordinary investment assumptions",
        "survey_basis": "Limited / purpose-specific",
        "transferability": "Restricted",
        "screening_view": (
            "Should not be treated as an ordinary private freehold investment title."
        )
    }
}


DEVELOPMENT_REGULATION_CHECKLIST = [
    "Land-use zoning and permitted use",
    "Building permits",
    "Building-control requirements",
    "Urban planning restrictions",
    "Environmental Impact Assessment (EIA), where applicable",
    "Height or density restrictions",
    "Access and right-of-way",
    "Local authority approvals",
    "Certificate of completion / occupation, where required",
    "Developer obligations and project registration"
]


CONSUMER_PROTECTION_CHECKLIST = [
    "Project details and unit specifications",
    "Purchase price and payment schedule",
    "Construction and completion dates",
    "Developer obligations",
    "Changes to plans or specifications",
    "Marketing representations versus delivered product",
    "Handover conditions",
    "Common-area transfer and juristic-person arrangements"
]


# ------------------------------------------------------------
# Property alias index
# ------------------------------------------------------------

PROPERTY_ALIAS_INDEX = {}

for property_type, data in PROPERTY_TYPES.items():

    PROPERTY_ALIAS_INDEX[property_type] = property_type

    for alias in data["aliases"]:
        PROPERTY_ALIAS_INDEX[alias.lower()] = property_type


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

legal_validation_errors = []

for rule_name, rule_data in LEGAL_RULES.items():

    required_rule_fields = [
        "status",
        "rule",
        "key_condition",
        "due_diligence"
    ]

    for field in required_rule_fields:
        if field not in rule_data:
            legal_validation_errors.append(
                f"{rule_name}: missing field '{field}'"
            )


if legal_validation_errors:

    print("⚠ Legal knowledge-base validation problems:")

    for error in legal_validation_errors:
        print("-", error)

else:

    print("✓ Legal & property knowledge base loaded successfully")
    print(f"✓ {len(PROPERTY_TYPES)} property categories loaded")
    print(f"✓ {len(LEGAL_RULES)} foreign-investor legal frameworks loaded")
    print(f"✓ {len(THAI_LAND_TITLES)} Thai land-title categories loaded")
    print(f"✓ {len(PROPERTY_ALIAS_INDEX)} property names / aliases indexed")
    print("✓ Legal knowledge-base validation passed")

✓ Legal & property knowledge base loaded successfully
✓ 6 property categories loaded
✓ 4 foreign-investor legal frameworks loaded
✓ 4 Thai land-title categories loaded
✓ 33 property names / aliases indexed
✓ Legal knowledge-base validation passed


In [4]:
# CELL 4 - Real Estate Investment & Valuation Knowledge Base

VALUATION_CONCEPTS = {

    "noi": {
        "name": "Net Operating Income (NOI)",
        "aliases": [
            "noi",
            "net operating income"
        ],
        "definition": (
            "NOI measures the income generated by a property after deducting "
            "operating expenses, but before financing costs, income taxes "
            "and depreciation."
        ),
        "formula": (
            "NOI = Effective Gross Income - Operating Expenses"
        ),
        "investment_use": (
            "NOI is a central input for income-producing property analysis "
            "and capitalization-based valuation."
        ),
        "caution": (
            "The quality of the result depends on realistic rent, occupancy "
            "and operating-expense assumptions."
        )
    },

    "cap_rate": {
        "name": "Capitalization Rate (Cap Rate)",
        "aliases": [
            "cap rate",
            "capitalization rate",
            "capitalisation rate"
        ],
        "definition": (
            "The cap rate relates a property's annual Net Operating Income "
            "to its value or purchase price."
        ),
        "formula": (
            "Cap Rate = NOI / Property Value"
        ),
        "investment_use": (
            "It provides a simple unlevered income-yield indicator and can "
            "also be used to capitalize stabilized NOI into an estimated value."
        ),
        "caution": (
            "Cap rates should be compared only with appropriate properties "
            "and markets and do not capture the full timing of future cash flows."
        )
    },

    "gross_rental_yield": {
        "name": "Gross Rental Yield",
        "aliases": [
            "gross rental yield",
            "gross yield",
            "rental yield",
            "yield"
        ],
        "definition": (
            "Gross rental yield compares annual gross rental income with the "
            "property purchase price before operating costs."
        ),
        "formula": (
            "Gross Rental Yield = Annual Gross Rent / Purchase Price"
        ),
        "investment_use": (
            "It is useful as a quick screening metric for rental-income potential."
        ),
        "caution": (
            "Gross yield is not the investor's net return because vacancy, "
            "common-area fees, maintenance, taxes, management and other costs "
            "are not deducted."
        )
    },

    "npv": {
        "name": "Net Present Value (NPV)",
        "aliases": [
            "npv",
            "net present value"
        ],
        "definition": (
            "NPV measures the present value of expected future cash flows "
            "minus the initial investment."
        ),
        "formula": (
            "NPV = Sum of discounted future cash flows - Initial Investment"
        ),
        "investment_use": (
            "A positive NPV indicates that the modeled investment creates "
            "value relative to the selected discount rate and assumptions."
        ),
        "caution": (
            "NPV is highly sensitive to assumptions about rent, vacancy, "
            "expenses, resale value, holding period and discount rate."
        )
    },

    "irr": {
        "name": "Internal Rate of Return (IRR)",
        "aliases": [
            "irr",
            "internal rate of return"
        ],
        "definition": (
            "IRR is the discount rate at which the NPV of the modeled "
            "investment cash flows equals zero."
        ),
        "formula": (
            "IRR is the rate r for which NPV(r) = 0."
        ),
        "investment_use": (
            "It is commonly used to compare expected returns across investment "
            "opportunities with multi-period cash flows."
        ),
        "caution": (
            "IRR should not be interpreted without reviewing the underlying "
            "cash flows, timing assumptions, reinvestment assumptions and risk."
        )
    },

    "income_approach": {
        "name": "Income Approach",
        "aliases": [
            "income approach",
            "income method",
            "income capitalization",
            "income capitalisation"
        ],
        "definition": (
            "The Income Approach estimates property value from the income "
            "the property is expected to generate."
        ),
        "formula": (
            "Simple direct capitalization: Value = NOI / Cap Rate"
        ),
        "investment_use": (
            "It is particularly relevant for income-producing real estate."
        ),
        "caution": (
            "Reliable NOI and market-supported capitalization or discount "
            "rates are essential."
        )
    },

    "cost_approach": {
        "name": "Cost Approach",
        "aliases": [
            "cost approach",
            "cost method",
            "replacement cost"
        ],
        "definition": (
            "The Cost Approach estimates value based on the cost of creating "
            "a comparable property, adjusted for depreciation and combined "
            "with land value where relevant."
        ),
        "formula": (
            "Indicative structure: Land Value + Replacement/Reproduction Cost "
            "- Depreciation"
        ),
        "investment_use": (
            "It can be useful for newer, specialized or less frequently traded "
            "properties where income or comparable evidence is limited."
        ),
        "caution": (
            "Estimated construction cost does not automatically equal market value."
        )
    },

    "sales_comparison": {
        "name": "Sales Comparison Approach",
        "aliases": [
            "sales comparison",
            "sales comparison approach",
            "market comparison",
            "comparable sales",
            "comps"
        ],
        "definition": (
            "The Sales Comparison Approach estimates value by comparing the "
            "subject property with relevant comparable market transactions."
        ),
        "formula": (
            "Value indication = Comparable transaction evidence adjusted for "
            "material differences."
        ),
        "investment_use": (
            "It is especially useful when sufficient recent and genuinely "
            "comparable transaction evidence is available."
        ),
        "caution": (
            "Asking prices are not equivalent to completed transaction prices, "
            "and poor comparables can produce misleading valuations."
        )
    }
}


INVESTMENT_FACTORS = {

    "location_market": {
        "name": "Location & Market Conditions",
        "description": (
            "Accessibility, employment centers, neighborhood quality, tenant "
            "demand, infrastructure and market supply-demand conditions can "
            "materially affect value and rental performance."
        )
    },

    "capital_finance": {
        "name": "Capital Availability & Finance",
        "description": (
            "Budget, financing structure, interest costs, transaction costs "
            "and liquidity constraints affect feasibility and returns."
        )
    },

    "regulation": {
        "name": "Regulatory Environment",
        "description": (
            "Ownership rules, land title, zoning, permits, foreign-investor "
            "restrictions and transaction requirements must be screened before "
            "financial attractiveness is treated as decisive."
        )
    },

    "management": {
        "name": "Management Involvement",
        "description": (
            "Different property types require different levels of leasing, "
            "maintenance, tenant management and operational involvement."
        )
    },

    "time_liquidity": {
        "name": "Time Horizon & Liquidity",
        "description": (
            "Holding period, resale demand and the time required to exit an "
            "investment influence its suitability for an investor."
        )
    },

    "return_risk": {
        "name": "Return Expectations & Risk Tolerance",
        "description": (
            "Income, capital growth and downside risk should be evaluated "
            "together rather than maximizing a single return metric."
        )
    }
}


DUE_DILIGENCE_CATEGORIES = {

    "legal": [
        "Verify title and ownership records.",
        "Confirm the legally permitted ownership structure.",
        "Check foreign-ownership restrictions where applicable.",
        "Review contracts, encumbrances, easements and rights affecting the property.",
        "Confirm required registrations and approvals."
    ],

    "financial": [
        "Verify the exact purchase price and transaction costs.",
        "Estimate realistic achievable rent rather than relying only on advertised rent.",
        "Check vacancy assumptions.",
        "Review common-area fees and recurring operating expenses.",
        "Stress-test income and resale assumptions."
    ],

    "market": [
        "Review recent comparable transaction evidence where available.",
        "Assess competing supply and development pipeline.",
        "Evaluate tenant demand.",
        "Assess resale liquidity.",
        "Compare the property with relevant alternative locations."
    ],

    "technical": [
        "Inspect building condition and maintenance history.",
        "Review structural and foundation information where relevant.",
        "Check flood history and drainage.",
        "Review access during heavy rainfall.",
        "Consider site-specific geotechnical or engineering due diligence where appropriate."
    ],

    "building_condo": [
        "Review condominium juristic-person financial statements.",
        "Check common-area fees and sinking-fund obligations.",
        "Review maintenance quality and major planned repairs.",
        "Confirm remaining foreign ownership quota where applicable.",
        "Assess unit layout, floor, orientation and building-specific rental demand."
    ],

    "exit": [
        "Assess likely buyer demand at resale.",
        "Review expected holding period.",
        "Consider transaction costs at exit.",
        "Avoid assuming future capital appreciation is guaranteed.",
        "Evaluate whether the investment remains acceptable under weaker resale conditions."
    ]
}


TECHNICAL_RISK_KNOWLEDGE = {

    "flood": {
        "name": "Flood & Drainage Screening",
        "aliases": [
            "flood",
            "flooding",
            "drainage",
            "heavy rain",
            "rainfall",
            "waterlogging"
        ],
        "checks": [
            "Check historical flooding at the specific street and project.",
            "Check soi and road access during heavy rainfall.",
            "Review local drainage conditions.",
            "Check building entrance, parking and critical equipment elevation.",
            "Ask building management about previous flood events and mitigation measures.",
            "Do not infer site-specific flood safety from an area-level score alone."
        ],
        "limitation": (
            "Flood exposure can vary materially within the same Bangkok district. "
            "Area-level screening cannot replace project- and street-level checks."
        )
    },

    "soil": {
        "name": "Soil, Foundation & Engineering Screening",
        "aliases": [
            "soil",
            "soft clay",
            "clay",
            "foundation",
            "geotechnical",
            "engineering",
            "subsidence",
            "settlement",
            "structural"
        ],
        "checks": [
            "Review available foundation and structural documentation.",
            "Consider Bangkok's soft-clay ground conditions in technical due diligence.",
            "Check for visible settlement, cracking or water-related deterioration.",
            "Review building age and maintenance history.",
            "For development or site-specific decisions, obtain qualified geotechnical and engineering advice.",
            "Do not infer structural safety from district-level soil information."
        ],
        "limitation": (
            "Area-level soil descriptions are screening indicators only and "
            "cannot determine the structural condition of an individual building."
        )
    }
}


# ------------------------------------------------------------
# Alias indices for later intent/concept detection
# ------------------------------------------------------------

VALUATION_ALIAS_INDEX = {}

for concept_key, concept_data in VALUATION_CONCEPTS.items():

    VALUATION_ALIAS_INDEX[concept_key.lower()] = concept_key

    for alias in concept_data["aliases"]:
        VALUATION_ALIAS_INDEX[alias.lower()] = concept_key


TECHNICAL_ALIAS_INDEX = {}

for risk_key, risk_data in TECHNICAL_RISK_KNOWLEDGE.items():

    TECHNICAL_ALIAS_INDEX[risk_key.lower()] = risk_key

    for alias in risk_data["aliases"]:
        TECHNICAL_ALIAS_INDEX[alias.lower()] = risk_key


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

valuation_errors = []

required_concept_fields = [
    "name",
    "aliases",
    "definition",
    "formula",
    "investment_use",
    "caution"
]

for concept_key, concept_data in VALUATION_CONCEPTS.items():

    for field in required_concept_fields:

        if field not in concept_data:
            valuation_errors.append(
                f"{concept_key}: missing field '{field}'"
            )


if valuation_errors:

    print("⚠ Investment / valuation knowledge-base validation problems:")

    for error in valuation_errors:
        print("-", error)

else:

    print("✓ Investment & valuation knowledge base loaded successfully")
    print(f"✓ {len(VALUATION_CONCEPTS)} valuation concepts loaded")
    print(f"✓ {len(INVESTMENT_FACTORS)} investment-decision factors loaded")
    print(f"✓ {len(DUE_DILIGENCE_CATEGORIES)} due-diligence categories loaded")
    print(f"✓ {len(TECHNICAL_RISK_KNOWLEDGE)} technical-risk modules loaded")
    print(f"✓ {len(VALUATION_ALIAS_INDEX)} valuation terms / aliases indexed")
    print(f"✓ {len(TECHNICAL_ALIAS_INDEX)} technical-risk terms / aliases indexed")
    print("✓ Investment / valuation knowledge-base validation passed")

✓ Investment & valuation knowledge base loaded successfully
✓ 8 valuation concepts loaded
✓ 6 investment-decision factors loaded
✓ 6 due-diligence categories loaded
✓ 2 technical-risk modules loaded
✓ 30 valuation terms / aliases indexed
✓ 15 technical-risk terms / aliases indexed
✓ Investment / valuation knowledge-base validation passed


In [5]:
# CELL 5 - Query Understanding & Entity Extraction Engine

import re
import unicodedata


# ============================================================
# 1. TEXT NORMALIZATION
# ============================================================

def normalize_text(text):
    if text is None:
        return ""

    text = str(text).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = text.replace("–", "-")
    text = text.replace("—", "-")
    text = text.replace("’", "'")
    text = re.sub(r"\s+", " ", text)

    return text


# ============================================================
# 2. FOREIGN INVESTOR DETECTION
# ============================================================

FOREIGN_INVESTOR_TERMS = [
    "foreigner",
    "foreign investor",
    "foreign buyer",
    "non thai",
    "non-thai",
    "expat",
    "international investor",
    "international buyer",
    "german",
    "french",
    "british",
    "american",
    "european",
    "chinese",
    "japanese",
    "korean",
    "singaporean",
    "australian"
]


def detect_foreign_investor(query):
    query = normalize_text(query)

    detected_terms = []

    for term in FOREIGN_INVESTOR_TERMS:
        if term in query:
            detected_terms.append(term)

    return {
        "is_foreign_context": bool(detected_terms),
        "matched_terms": detected_terms
    }


# ============================================================
# 3. PROPERTY TYPE DETECTION
# ============================================================

def detect_property_types(query):
    query = normalize_text(query)

    matches = []

    sorted_aliases = sorted(
        PROPERTY_ALIAS_INDEX.items(),
        key=lambda x: len(x[0]),
        reverse=True
    )

    for alias, property_type in sorted_aliases:
        alias_normalized = normalize_text(alias)

        pattern = r"(?<!\w)" + re.escape(alias_normalized) + r"(?!\w)"

        if re.search(pattern, query):
            if property_type not in matches:
                matches.append(property_type)

    return matches


# ============================================================
# 4. PURCHASE STRUCTURE DETECTION
# ============================================================

FREEHOLD_TERMS = [
    "freehold",
    "own outright",
    "direct ownership",
    "own the land",
    "buy the land",
    "own land",
    "ownership"
]

LEASEHOLD_TERMS = [
    "leasehold",
    "lease",
    "leasing",
    "long term lease",
    "long-term lease",
    "rent the land"
]


def detect_purchase_structures(query):
    query = normalize_text(query)

    structures = []

    if any(term in query for term in LEASEHOLD_TERMS):
        structures.append("leasehold")

    if any(term in query for term in FREEHOLD_TERMS):
        structures.append("freehold")

    return structures


# ============================================================
# 5. LOCATION DETECTION
# ============================================================

def detect_locations(query):
    query = normalize_text(query)

    found_zones = []

    sorted_locations = sorted(
        ZONE_ALIAS_INDEX.items(),
        key=lambda x: len(x[0]),
        reverse=True
    )

    for alias, canonical_zone in sorted_locations:
        alias_normalized = normalize_text(alias)

        pattern = r"(?<!\w)" + re.escape(alias_normalized) + r"(?!\w)"

        if re.search(pattern, query):
            if canonical_zone not in found_zones:
                found_zones.append(canonical_zone)

    return found_zones


# ============================================================
# 6. BUDGET EXTRACTION
# ============================================================

def extract_budget_million_thb(query):
    query = normalize_text(query)

    patterns = [
        r"(?:thb\s*)?(\d+(?:\.\d+)?)\s*(million|m)\s*(?:thb|baht)?",
        r"(?:thb|baht)\s*(\d+(?:\.\d+)?)\s*(million|m)",
        r"(\d{1,3}(?:,\d{3})+)\s*(?:thb|baht)"
    ]

    for pattern in patterns:
        match = re.search(pattern, query)

        if match:
            raw_value = match.group(1).replace(",", "")

            try:
                value = float(raw_value)
            except ValueError:
                continue

            if value >= 100000:
                return round(value / 1_000_000, 3)

            return value

    return None


# ============================================================
# 7. PROPERTY SIZE EXTRACTION
# ============================================================

def extract_property_size_sqm(query):
    query = normalize_text(query)

    patterns = [
        r"(\d+(?:\.\d+)?)\s*(?:sqm|sq m|m2|m²|square meters|square metres)",
        r"(\d+(?:\.\d+)?)\s*(?:square meter|square metre)"
    ]

    for pattern in patterns:
        match = re.search(pattern, query)

        if match:
            try:
                return float(match.group(1))
            except ValueError:
                pass

    return None


# ============================================================
# 8. INVESTMENT STRATEGY DETECTION
# ============================================================

STRATEGY_TERMS = {
    "income": [
        "rental income",
        "income",
        "cash flow",
        "cashflow",
        "rent",
        "rental",
        "yield",
        "high yield",
        "passive income"
    ],

    "growth": [
        "growth",
        "capital growth",
        "appreciation",
        "capital appreciation",
        "price growth",
        "future value",
        "upside"
    ],

    "low_risk": [
        "low risk",
        "low-risk",
        "safe investment",
        "conservative",
        "stable",
        "stability",
        "risk averse",
        "risk-averse"
    ],

    "balanced": [
        "balanced",
        "balance",
        "mix of income and growth",
        "income and growth",
        "yield and growth"
    ]
}


def detect_investment_strategies(query):
    query = normalize_text(query)

    detected = []

    for strategy, terms in STRATEGY_TERMS.items():
        if any(term in query for term in terms):
            detected.append(strategy)

    return detected


# ============================================================
# 9. VALUATION CONCEPT DETECTION
# ============================================================

def detect_valuation_concepts(query):
    query = normalize_text(query)

    detected = []

    sorted_aliases = sorted(
        VALUATION_ALIAS_INDEX.items(),
        key=lambda x: len(x[0]),
        reverse=True
    )

    for alias, concept in sorted_aliases:
        alias_normalized = normalize_text(alias)

        pattern = r"(?<!\w)" + re.escape(alias_normalized) + r"(?!\w)"

        if re.search(pattern, query):
            if concept not in detected:
                detected.append(concept)

    return detected


# ============================================================
# 10. TECHNICAL RISK DETECTION
# ============================================================

def detect_technical_risks(query):
    query = normalize_text(query)

    detected = []

    sorted_aliases = sorted(
        TECHNICAL_ALIAS_INDEX.items(),
        key=lambda x: len(x[0]),
        reverse=True
    )

    for alias, risk in sorted_aliases:
        alias_normalized = normalize_text(alias)

        pattern = r"(?<!\w)" + re.escape(alias_normalized) + r"(?!\w)"

        if re.search(pattern, query):
            if risk not in detected:
                detected.append(risk)

    return detected


# ============================================================
# 11. QUESTION SIGNAL DETECTION
# ============================================================

QUESTION_SIGNALS = {
    "comparison": [
        "compare",
        "comparison",
        "versus",
        " vs ",
        "better than",
        "difference between"
    ],

    "recommendation": [
        "recommend",
        "best area",
        "best areas",
        "where should",
        "where can",
        "which area",
        "which areas",
        "where to invest"
    ],

    "legal": [
        "can a foreigner",
        "can i buy",
        "can i own",
        "allowed",
        "legal",
        "legally",
        "ownership",
        "foreign ownership",
        "quota",
        "land title",
        "chanote",
        "nor sor",
        "sor por gor",
        "leasehold",
        "freehold"
    ],

    "definition": [
        "what is",
        "what does",
        "define",
        "meaning of",
        "explain"
    ],

    "calculation": [
        "calculate",
        "calculation",
        "compute",
        "how much",
        "formula"
    ],

    "due_diligence": [
        "what should i check",
        "due diligence",
        "before buying",
        "before investing",
        "risk",
        "risks",
        "checklist"
    ]
}


def detect_question_signals(query):
    query = normalize_text(query)

    signals = []

    for signal, terms in QUESTION_SIGNALS.items():
        if any(term in query for term in terms):
            signals.append(signal)

    return signals


# ============================================================
# 12. MASTER QUERY PARSER
# ============================================================

def parse_investment_query(user_query):
    normalized_query = normalize_text(user_query)

    parsed = {
        "raw_query": user_query,
        "normalized_query": normalized_query,

        "foreign_investor": detect_foreign_investor(
            normalized_query
        ),

        "property_types": detect_property_types(
            normalized_query
        ),

        "purchase_structures": detect_purchase_structures(
            normalized_query
        ),

        "locations": detect_locations(
            normalized_query
        ),

        "budget_million_thb": extract_budget_million_thb(
            normalized_query
        ),

        "property_size_sqm": extract_property_size_sqm(
            normalized_query
        ),

        "investment_strategies": detect_investment_strategies(
            normalized_query
        ),

        "valuation_concepts": detect_valuation_concepts(
            normalized_query
        ),

        "technical_risks": detect_technical_risks(
            normalized_query
        ),

        "question_signals": detect_question_signals(
            normalized_query
        )
    }

    return parsed


# ============================================================
# 13. HUMAN-READABLE PARSER OUTPUT
# ============================================================

def print_query_parse(parsed):
    print("QUERY UNDERSTANDING")
    print("-" * 55)

    print("Query:")
    print(parsed["raw_query"])

    print("\nDetected:")

    print(
        "Foreign-investor context:",
        parsed["foreign_investor"]["is_foreign_context"]
    )

    print(
        "Property types:",
        parsed["property_types"]
    )

    print(
        "Purchase structures:",
        parsed["purchase_structures"]
    )

    print(
        "Locations:",
        parsed["locations"]
    )

    print(
        "Budget (million THB):",
        parsed["budget_million_thb"]
    )

    print(
        "Property size (sqm):",
        parsed["property_size_sqm"]
    )

    print(
        "Investment strategies:",
        parsed["investment_strategies"]
    )

    print(
        "Valuation concepts:",
        parsed["valuation_concepts"]
    )

    print(
        "Technical risks:",
        parsed["technical_risks"]
    )

    print(
        "Question signals:",
        parsed["question_signals"]
    )


print("✓ Query understanding & entity extraction engine loaded successfully")
print("✓ Master parser 'parse_investment_query' loaded successfully")

✓ Query understanding & entity extraction engine loaded successfully
✓ Master parser 'parse_investment_query' loaded successfully


In [6]:
# CELL 5A - Parser Stress Test

test_queries = [
    "I am a German investor with THB 6 million looking for a 45 sqm condo in On Nut with good rental income. What should I check regarding flooding?",
    "Can a foreigner buy land in Bangkok?",
    "Compare Bang Kapi with On Nut for rental income.",
    "What is the difference between gross rental yield and cap rate?",
    "I want a leasehold house in Bangkok. What should I know?",
    "What should I check regarding flood and soil conditions?",
    "Which Bangkok areas are best for capital growth?",
    "How do I calculate NPV for a real estate investment?"
]

for i, question in enumerate(test_queries, 1):
    print("\n" + "=" * 80)
    print(f"TEST {i}")
    print("=" * 80)

    result = parse_investment_query(question)
    print_query_parse(result)


TEST 1
QUERY UNDERSTANDING
-------------------------------------------------------
Query:
I am a German investor with THB 6 million looking for a 45 sqm condo in On Nut with good rental income. What should I check regarding flooding?

Detected:
Foreign-investor context: True
Property types: ['condominium']
Purchase structures: []
Locations: ['On Nut - Phra Khanong']
Budget (million THB): 6.0
Property size (sqm): 45.0
Investment strategies: ['income']
Valuation concepts: []
Technical risks: ['flood']
Question signals: ['due_diligence']

TEST 2
QUERY UNDERSTANDING
-------------------------------------------------------
Query:
Can a foreigner buy land in Bangkok?

Detected:
Foreign-investor context: True
Property types: ['land']
Purchase structures: []
Locations: []
Budget (million THB): None
Property size (sqm): None
Investment strategies: []
Valuation concepts: []
Technical risks: []
Question signals: ['legal']

TEST 3
QUERY UNDERSTANDING
-----------------------------------------------

In [7]:
# CELL 6 - Intent Routing & Decision Engine
# Converts parsed user questions into a clear primary response route.

# ============================================================
# 1. ROUTE DEFINITIONS
# ============================================================

RESPONSE_ROUTES = {
    "legal_restriction": {
        "label": "Legal / Regulatory Restriction",
        "description": "Foreign-investor ownership or acquisition restrictions."
    },

    "legal_guidance": {
        "label": "Legal / Ownership Guidance",
        "description": "Foreign ownership, leasehold, title and transaction guidance."
    },

    "area_comparison": {
        "label": "Bangkok Area Comparison",
        "description": "Direct comparison of two or more Bangkok investment areas."
    },

    "area_recommendation": {
        "label": "Bangkok Area Recommendation",
        "description": "Ranking or recommendation of Bangkok investment areas."
    },

    "property_screening": {
        "label": "Property / Area Investment Screening",
        "description": "Investment screening using property, location and investor inputs."
    },

    "technical_due_diligence": {
        "label": "Technical Due Diligence",
        "description": "Flood, drainage, soil, engineering or other technical risks."
    },

    "valuation_definition": {
        "label": "Valuation Concept Explanation",
        "description": "Explanation or comparison of valuation concepts."
    },

    "valuation_calculation": {
        "label": "Valuation Calculation",
        "description": "Calculation or formula-based valuation question."
    },

    "general_due_diligence": {
        "label": "Investment Due Diligence",
        "description": "General pre-investment checks and risk assessment."
    },

    "clarification": {
        "label": "Clarification Required",
        "description": "The request needs more information before reliable screening."
    },

    "general_real_estate": {
        "label": "General Real Estate Guidance",
        "description": "General real-estate question within prototype scope."
    }
}


# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def _has_any(values):
    return bool(values)


def _contains(parsed, field, value):
    return value in parsed.get(field, [])


def _is_foreign_context(parsed):
    return parsed.get(
        "foreign_investor", {}
    ).get(
        "is_foreign_context", False
    )


# ============================================================
# 3. LEGAL SEVERITY CHECK
# ============================================================

def detect_major_legal_constraint(parsed):
    """
    Detect cases where normal investment ranking should be stopped
    until the ownership structure has been legally clarified.
    """

    property_types = parsed.get("property_types", [])
    structures = parsed.get("purchase_structures", [])
    signals = parsed.get("question_signals", [])

    # Direct foreign land-purchase questions are regulation-first.
    if "land" in property_types and (
        _is_foreign_context(parsed) or "legal" in signals
    ):
        return {
            "major_constraint": True,
            "reason": (
                "Foreign individual land ownership in Thailand requires "
                "regulation-first screening before investment ranking."
            )
        }

    # House / townhouse ownership questions also require special
    # treatment because ownership of the structure and underlying
    # land must be distinguished.
    if (
        any(
            p in property_types
            for p in ["house", "townhouse"]
        )
        and "freehold" in structures
        and (
            _is_foreign_context(parsed)
            or "legal" in signals
        )
    ):
        return {
            "major_constraint": True,
            "reason": (
                "Foreign ownership of a house must be separated from "
                "ownership of the underlying land."
            )
        }

    return {
        "major_constraint": False,
        "reason": None
    }


# ============================================================
# 4. PRIMARY INTENT ROUTER
# ============================================================

def route_investment_query(parsed):
    """
    Determine the most appropriate primary response route.

    Priority logic:
    1. Major legal constraints
    2. Explicit legal questions
    3. Multi-location comparisons
    4. Valuation calculations
    5. Valuation explanations
    6. Technical due diligence
    7. General due diligence
    8. Area recommendations
    9. Property / area screening
    10. General guidance / clarification
    """

    locations = parsed.get("locations", [])
    property_types = parsed.get("property_types", [])
    structures = parsed.get("purchase_structures", [])
    strategies = parsed.get("investment_strategies", [])
    valuation_concepts = parsed.get("valuation_concepts", [])
    technical_risks = parsed.get("technical_risks", [])
    signals = parsed.get("question_signals", [])

    legal_constraint = detect_major_legal_constraint(parsed)

    # --------------------------------------------------------
    # ROUTE 1 - MAJOR LEGAL CONSTRAINT
    # --------------------------------------------------------

    if legal_constraint["major_constraint"]:
        return {
            "primary_route": "legal_restriction",
            "secondary_routes": [],
            "reason": legal_constraint["reason"],
            "confidence": "high",
            "needs_clarification": False
        }

    # --------------------------------------------------------
    # ROUTE 2 - LEGAL / OWNERSHIP QUESTION
    # --------------------------------------------------------

    if "legal" in signals or _has_any(structures):
        return {
            "primary_route": "legal_guidance",
            "secondary_routes": (
                ["property_screening"]
                if _has_any(locations)
                else []
            ),
            "reason": (
                "The question contains an ownership, leasehold, "
                "freehold or foreign-investor legal signal."
            ),
            "confidence": "high",
            "needs_clarification": False
        }

    # --------------------------------------------------------
    # ROUTE 3 - AREA COMPARISON
    # --------------------------------------------------------

    if (
        len(locations) >= 2
        or (
            "comparison" in signals
            and len(locations) >= 2
        )
    ):
        return {
            "primary_route": "area_comparison",
            "secondary_routes": (
                ["technical_due_diligence"]
                if _has_any(technical_risks)
                else []
            ),
            "reason": (
                f"{len(locations)} Bangkok areas were detected "
                "and should be compared directly."
            ),
            "confidence": "high",
            "needs_clarification": False
        }

    # --------------------------------------------------------
    # ROUTE 4 - VALUATION CALCULATION
    # --------------------------------------------------------

    if (
        "calculation" in signals
        and _has_any(valuation_concepts)
    ):
        return {
            "primary_route": "valuation_calculation",
            "secondary_routes": [],
            "reason": (
                "The question requests a calculation involving "
                "a recognized valuation concept."
            ),
            "confidence": "high",
            "needs_clarification": False
        }

    # --------------------------------------------------------
    # ROUTE 5 - VALUATION EXPLANATION / COMPARISON
    # --------------------------------------------------------

    if _has_any(valuation_concepts):
        return {
            "primary_route": "valuation_definition",
            "secondary_routes": [],
            "reason": (
                "The question refers to one or more recognized "
                "real-estate valuation concepts."
            ),
            "confidence": "high",
            "needs_clarification": False
        }

    # --------------------------------------------------------
    # ROUTE 6 - TECHNICAL DUE DILIGENCE
    # --------------------------------------------------------

    if _has_any(technical_risks):
        secondary = []

        if _has_any(locations):
            secondary.append("property_screening")

        return {
            "primary_route": "technical_due_diligence",
            "secondary_routes": secondary,
            "reason": (
                "The question explicitly asks about technical "
                "property or location risks."
            ),
            "confidence": "high",
            "needs_clarification": False
        }

    # --------------------------------------------------------
    # ROUTE 7 - GENERAL DUE DILIGENCE
    # --------------------------------------------------------

    if "due_diligence" in signals:
        return {
            "primary_route": "general_due_diligence",
            "secondary_routes": [],
            "reason": (
                "The user asks what should be checked before "
                "buying or investing."
            ),
            "confidence": "high",
            "needs_clarification": False
        }

    # --------------------------------------------------------
    # ROUTE 8 - AREA RECOMMENDATION
    # --------------------------------------------------------

    if (
        "recommendation" in signals
        or (
            _has_any(strategies)
            and not _has_any(locations)
        )
    ):
        return {
            "primary_route": "area_recommendation",
            "secondary_routes": [],
            "reason": (
                "The question asks for Bangkok investment-area "
                "recommendations or rankings."
            ),
            "confidence": "high",
            "needs_clarification": False
        }

    # --------------------------------------------------------
    # ROUTE 9 - PROPERTY / AREA SCREENING
    # --------------------------------------------------------

    if (
        _has_any(locations)
        or _has_any(property_types)
        or parsed.get("budget_million_thb") is not None
        or parsed.get("property_size_sqm") is not None
    ):
        return {
            "primary_route": "property_screening",
            "secondary_routes": [],
            "reason": (
                "The question contains concrete property, location, "
                "budget or size information suitable for screening."
            ),
            "confidence": "medium",
            "needs_clarification": False
        }

    # --------------------------------------------------------
    # ROUTE 10 - GENERAL REAL ESTATE / CLARIFICATION
    # --------------------------------------------------------

    normalized_query = parsed.get("normalized_query", "")

    if len(normalized_query.split()) >= 4:
        return {
            "primary_route": "general_real_estate",
            "secondary_routes": [],
            "reason": (
                "The question is within the broader real-estate "
                "domain but does not match a specialized route."
            ),
            "confidence": "medium",
            "needs_clarification": False
        }

    return {
        "primary_route": "clarification",
        "secondary_routes": [],
        "reason": (
            "The request does not contain enough information "
            "for reliable intent classification."
        ),
        "confidence": "low",
        "needs_clarification": True
    }


# ============================================================
# 5. HUMAN-READABLE ROUTER OUTPUT
# ============================================================

def print_route_decision(question, parsed, route):
    route_key = route["primary_route"]

    route_label = RESPONSE_ROUTES.get(
        route_key, {}
    ).get(
        "label", route_key
    )

    print("INTENT ROUTING")
    print("-" * 55)
    print("Question:")
    print(question)

    print("\nPrimary route:")
    print(f"{route_key} -> {route_label}")

    print("\nSecondary routes:")
    print(route["secondary_routes"])

    print("\nReason:")
    print(route["reason"])

    print("\nConfidence:")
    print(route["confidence"])

    print("\nNeeds clarification:")
    print(route["needs_clarification"])


print("✓ Intent routing & decision engine loaded successfully")
print(f"✓ {len(RESPONSE_ROUTES)} response routes available")

✓ Intent routing & decision engine loaded successfully
✓ 11 response routes available


In [8]:
# CELL 6A - Intent Router Stress Test

router_test_queries = [
    "I am a German investor with THB 6 million looking for a 45 sqm condo in On Nut with good rental income. What should I check regarding flooding?",
    "Can a foreigner buy land in Bangkok?",
    "Compare Bang Kapi with On Nut for rental income.",
    "What is the difference between gross rental yield and cap rate?",
    "I want a leasehold house in Bangkok. What should I know?",
    "What should I check regarding flood and soil conditions?",
    "Which Bangkok areas are best for capital growth?",
    "How do I calculate NPV for a real estate investment?"
]

expected_routes = [
    "technical_due_diligence",
    "legal_restriction",
    "area_comparison",
    "valuation_definition",
    "legal_guidance",
    "technical_due_diligence",
    "area_recommendation",
    "valuation_calculation"
]

passed = 0

for i, (question, expected) in enumerate(
    zip(router_test_queries, expected_routes), 1
):
    parsed = parse_investment_query(question)
    route = route_investment_query(parsed)

    actual = route["primary_route"]
    success = actual == expected

    if success:
        passed += 1

    print("\n" + "=" * 80)
    print(f"TEST {i}")
    print("=" * 80)
    print("Question:")
    print(question)

    print("\nExpected route:")
    print(expected)

    print("\nActual route:")
    print(actual)

    print("\nResult:")
    print("✓ PASS" if success else "✗ FAIL")

    print("\nReason:")
    print(route["reason"])

print("\n" + "=" * 80)
print("ROUTER STRESS TEST SUMMARY")
print("=" * 80)
print(f"Passed: {passed}/{len(router_test_queries)}")

if passed == len(router_test_queries):
    print("✓ ALL ROUTER TESTS PASSED")
else:
    print(
        f"⚠ {len(router_test_queries) - passed} "
        "router test(s) require adjustment"
    )


TEST 1
Question:
I am a German investor with THB 6 million looking for a 45 sqm condo in On Nut with good rental income. What should I check regarding flooding?

Expected route:
technical_due_diligence

Actual route:
technical_due_diligence

Result:
✓ PASS

Reason:
The question explicitly asks about technical property or location risks.

TEST 2
Question:
Can a foreigner buy land in Bangkok?

Expected route:
legal_restriction

Actual route:
legal_restriction

Result:
✓ PASS

Reason:
Foreign individual land ownership in Thailand requires regulation-first screening before investment ranking.

TEST 3
Question:
Compare Bang Kapi with On Nut for rental income.

Expected route:
area_comparison

Actual route:
area_comparison

Result:
✓ PASS

Reason:
2 Bangkok areas were detected and should be compared directly.

TEST 4
Question:
What is the difference between gross rental yield and cap rate?

Expected route:
valuation_definition

Actual route:
valuation_definition

Result:
✓ PASS

Reason:
The

In [9]:
# CELL 7A - Valuation & Definition Response Engine

def format_bullet_list(items):
    if not items:
        return ""

    return "\n".join(
        f"- {item}"
        for item in items
    )


def format_concept_block(concept_key):
    """
    Build a structured explanation for one valuation concept.
    """

    concept = VALUATION_CONCEPTS.get(concept_key)

    if concept is None:
        return (
            "The requested valuation concept is not currently "
            "available in the structured knowledge base."
        )

    lines = []

    lines.append(
        f"### {concept['name']}"
    )

    lines.append("")

    lines.append(
        f"**Definition:** {concept['definition']}"
    )

    lines.append("")

    lines.append(
        f"**Formula / structure:** {concept['formula']}"
    )

    lines.append("")

    lines.append(
        f"**Investment use:** {concept['investment_use']}"
    )

    lines.append("")

    lines.append(
        f"**Important caution:** {concept['caution']}"
    )

    return "\n".join(lines)


def build_valuation_definition_response(parsed):
    """
    Answer definition, explanation and concept-comparison questions.
    """

    concepts = parsed.get(
        "valuation_concepts",
        []
    )

    response = []

    response.append(
        "# 🇹🇭 Bangkok Foreign Investor AI"
    )

    response.append("")

    response.append(
        "## Valuation Concept Explanation"
    )

    response.append("")

    if not concepts:

        response.append(
            "I could not identify a specific valuation concept "
            "in the question."
        )

        response.append("")

        response.append(
            "The current valuation knowledge base covers "
            "**NOI, Cap Rate, Gross Rental Yield, NPV, IRR, "
            "Income Approach, Cost Approach and Sales Comparison**."
        )

        response.append("")

        response.append(
            "Please specify the concept you would like explained."
        )

        response.append("")

        response.append(
            f"**Model limitation:** {DATA_LIMITATION_NOTE}"
        )

        response.append("")

        response.append(
            f"**Disclaimer:** {LEGAL_DISCLAIMER}"
        )

        return "\n".join(response)

    if len(concepts) == 1:

        concept_key = concepts[0]

        response.append(
            format_concept_block(
                concept_key
            )
        )

    else:

        response.append(
            f"The question refers to **{len(concepts)} "
            "valuation concepts**. They are explained below."
        )

        response.append("")

        for concept_key in concepts:

            response.append(
                format_concept_block(
                    concept_key
                )
            )

            response.append("")

        response.append(
            "## Key Difference"
        )

        response.append("")

        if (
            "gross_rental_yield" in concepts
            and "cap_rate" in concepts
        ):

            response.append(
                "**Gross rental yield** uses gross rent before "
                "operating expenses, while **Cap Rate** uses "
                "Net Operating Income (NOI)."
            )

            response.append("")

            response.append(
                "Therefore, gross rental yield is a quick screening "
                "metric, whereas Cap Rate is generally a more refined "
                "income-property metric because operating expenses "
                "are incorporated through NOI."
            )

        elif (
            "npv" in concepts
            and "irr" in concepts
        ):

            response.append(
                "**NPV** expresses modeled value creation in currency "
                "terms at a chosen discount rate, while **IRR** is the "
                "discount rate that makes modeled NPV equal to zero."
            )

            response.append("")

            response.append(
                "Both depend heavily on the quality of the underlying "
                "cash-flow assumptions."
            )

        else:

            response.append(
                "These concepts serve different purposes. "
                "They should not be treated as interchangeable without "
                "considering the valuation context and available data."
            )

    response.append("")

    response.append(
        "## Investment Interpretation"
    )

    response.append("")

    response.append(
        "A valuation metric should normally be interpreted together "
        "with market evidence, legal feasibility, property condition, "
        "risk and the investor's holding period rather than used as "
        "a standalone decision rule."
    )

    response.append("")

    response.append(
        f"**Model limitation:** {DATA_LIMITATION_NOTE}"
    )

    response.append("")

    response.append(
        f"**Disclaimer:** {LEGAL_DISCLAIMER}"
    )

    return "\n".join(response)


# ============================================================
# VALUATION CALCULATION RESPONSE
# ============================================================

def build_valuation_calculation_response(parsed):
    """
    Explain how to calculate recognized valuation concepts.
    If the query does not provide enough numeric information,
    identify the required inputs instead of inventing values.
    """

    concepts = parsed.get(
        "valuation_concepts",
        []
    )

    response = []

    response.append(
        "# 🇹🇭 Bangkok Foreign Investor AI"
    )

    response.append("")

    response.append(
        "## Valuation Calculation Guidance"
    )

    response.append("")

    if not concepts:

        response.append(
            "I could not identify which valuation calculation "
            "you want to perform."
        )

        response.append("")

        response.append(
            "Please specify a concept such as **NPV, IRR, NOI, "
            "Cap Rate, Income Approach or Cost Approach**."
        )

        response.append("")

        response.append(
            f"**Disclaimer:** {LEGAL_DISCLAIMER}"
        )

        return "\n".join(response)

    primary_concept = concepts[0]

    concept = VALUATION_CONCEPTS.get(
        primary_concept
    )

    response.append(
        f"### {concept['name']}"
    )

    response.append("")

    response.append(
        f"**Formula / structure:** {concept['formula']}"
    )

    response.append("")

    # --------------------------------------------------------
    # NPV
    # --------------------------------------------------------

    if primary_concept == "npv":

        response.append(
            "To calculate NPV for a real-estate investment, "
            "you normally need:"
        )

        response.append("")

        response.append(
            format_bullet_list([
                "Initial investment / purchase price including relevant transaction costs",
                "Expected periodic net cash flows",
                "Holding period",
                "Expected sale or terminal proceeds",
                "Selling costs",
                "Discount rate / required rate of return"
            ])
        )

        response.append("")

        response.append(
            "The simplified logic is:"
        )

        response.append("")

        response.append(
            "**NPV = Present value of future net cash flows "
            "+ present value of terminal proceeds "
            "- initial investment.**"
        )

        response.append("")

        response.append(
            "If the modeled NPV is positive, the investment "
            "creates value relative to the selected discount rate "
            "under the stated assumptions. A negative NPV suggests "
            "the modeled return does not meet that required return."
        )

    # --------------------------------------------------------
    # IRR
    # --------------------------------------------------------

    elif primary_concept == "irr":

        response.append(
            "To calculate IRR, you need the complete sequence "
            "of investment cash flows:"
        )

        response.append("")

        response.append(
            format_bullet_list([
                "Initial cash outflow",
                "Periodic net operating or investment cash flows",
                "Final-year sale / terminal proceeds",
                "Selling costs"
            ])
        )

        response.append("")

        response.append(
            "IRR is the discount rate at which the NPV of those "
            "cash flows equals zero."
        )

        response.append("")

        response.append(
            "In practice it is normally solved numerically rather "
            "than by a simple closed-form formula."
        )

    # --------------------------------------------------------
    # NOI
    # --------------------------------------------------------

    elif primary_concept == "noi":

        response.append(
            "The simplified calculation is:"
        )

        response.append("")

        response.append(
            "**NOI = Effective Gross Income - Operating Expenses**"
        )

        response.append("")

        response.append(
            "Typical inputs include:"
        )

        response.append("")

        response.append(
            format_bullet_list([
                "Potential rental income",
                "Occupancy / vacancy",
                "Other operating income where relevant",
                "Property operating expenses",
                "Maintenance and management costs where applicable"
            ])
        )

        response.append("")

        response.append(
            "Debt service, income tax and depreciation are normally "
            "not deducted when calculating property-level NOI."
        )

    # --------------------------------------------------------
    # CAP RATE
    # --------------------------------------------------------

    elif primary_concept == "cap_rate":

        response.append(
            "The simplified calculation is:"
        )

        response.append("")

        response.append(
            "**Cap Rate = Annual NOI / Property Value**"
        )

        response.append("")

        response.append(
            "Example: if annual NOI is THB 300,000 and the "
            "property value is THB 6,000,000:"
        )

        response.append("")

        response.append(
            "**Cap Rate = 300,000 / 6,000,000 = 5.0%**"
        )

    # --------------------------------------------------------
    # GROSS RENTAL YIELD
    # --------------------------------------------------------

    elif primary_concept == "gross_rental_yield":

        response.append(
            "The simplified calculation is:"
        )

        response.append("")

        response.append(
            "**Gross Rental Yield = Annual Gross Rent / Purchase Price**"
        )

        response.append("")

        response.append(
            "Example: monthly rent of THB 25,000 produces annual "
            "gross rent of THB 300,000. With a THB 5,000,000 purchase "
            "price:"
        )

        response.append("")

        response.append(
            "**Gross Rental Yield = 300,000 / 5,000,000 = 6.0%**"
        )

        response.append("")

        response.append(
            "This does not deduct vacancy, maintenance, management, "
            "common-area fees, tax or other operating costs."
        )

    # --------------------------------------------------------
    # INCOME APPROACH
    # --------------------------------------------------------

    elif primary_concept == "income_approach":

        response.append(
            "For a simple direct-capitalization analysis:"
        )

        response.append("")

        response.append(
            "**Value = NOI / Cap Rate**"
        )

        response.append("")

        response.append(
            "Example: annual NOI of THB 360,000 with a market-supported "
            "cap rate of 6% implies:"
        )

        response.append("")

        response.append(
            "**Value = 360,000 / 0.06 = THB 6,000,000**"
        )

        response.append("")

        response.append(
            "The cap rate should be market-supported and appropriate "
            "for the property's risk and market segment."
        )

    # --------------------------------------------------------
    # COST APPROACH
    # --------------------------------------------------------

    elif primary_concept == "cost_approach":

        response.append(
            "A simplified Cost Approach structure is:"
        )

        response.append("")

        response.append(
            "**Property Value = Land Value + Replacement/Reproduction "
            "Cost - Depreciation**"
        )

        response.append("")

        response.append(
            "Typical inputs include:"
        )

        response.append("")

        response.append(
            format_bullet_list([
                "Land value",
                "Building size",
                "Replacement or reproduction cost per sqm",
                "Building age",
                "Useful life",
                "Physical, functional and external depreciation"
            ])
        )

    # --------------------------------------------------------
    # SALES COMPARISON
    # --------------------------------------------------------

    elif primary_concept == "sales_comparison":

        response.append(
            "The Sales Comparison Approach requires relevant comparable "
            "transactions and adjustments for differences between the "
            "subject property and the comparables."
        )

        response.append("")

        response.append(
            "Typical comparison variables include:"
        )

        response.append("")

        response.append(
            format_bullet_list([
                "Location",
                "Property size",
                "Building age and condition",
                "Floor / view / orientation",
                "Transaction date",
                "Unit characteristics",
                "Access and amenities"
            ])
        )

        response.append("")

        response.append(
            "Completed transaction evidence is generally more reliable "
            "than asking prices."
        )

    else:

        response.append(
            concept["definition"]
        )

        response.append("")

        response.append(
            f"**Important caution:** {concept['caution']}"
        )

    response.append("")

    response.append(
        "## Inputs & Assumptions"
    )

    response.append("")

    response.append(
        "A reliable property-specific calculation requires actual "
        "property inputs. The chatbot should not invent missing rents, "
        "costs, cap rates, resale values or discount rates."
    )

    response.append("")

    response.append(
        "## Model Limitation"
    )

    response.append("")

    response.append(
        DATA_LIMITATION_NOTE
    )

    response.append("")

    response.append(
        "## Disclaimer"
    )

    response.append("")

    response.append(
        LEGAL_DISCLAIMER
    )

    return "\n".join(response)


print("✓ Valuation & definition response engine loaded successfully")

✓ Valuation & definition response engine loaded successfully


In [10]:
# CELL 7A TEST - Valuation Response Quality

test_questions = [
    "What is the difference between gross rental yield and cap rate?",
    "How do I calculate NPV for a real estate investment?"
]

for i, question in enumerate(test_questions, 1):
    print("\n" + "=" * 90)
    print(f"TEST {i}: {question}")
    print("=" * 90)

    parsed = parse_investment_query(question)
    route = route_investment_query(parsed)

    print(f"Detected route: {route['primary_route']}")
    print()

    if route["primary_route"] == "valuation_definition":
        answer = build_valuation_definition_response(parsed)

    elif route["primary_route"] == "valuation_calculation":
        answer = build_valuation_calculation_response(parsed)

    else:
        answer = f"Unexpected route: {route['primary_route']}"

    display(Markdown(answer))


TEST 1: What is the difference between gross rental yield and cap rate?
Detected route: valuation_definition



# 🇹🇭 Bangkok Foreign Investor AI

## Valuation Concept Explanation

The question refers to **2 valuation concepts**. They are explained below.

### Gross Rental Yield

**Definition:** Gross rental yield compares annual gross rental income with the property purchase price before operating costs.

**Formula / structure:** Gross Rental Yield = Annual Gross Rent / Purchase Price

**Investment use:** It is useful as a quick screening metric for rental-income potential.

**Important caution:** Gross yield is not the investor's net return because vacancy, common-area fees, maintenance, taxes, management and other costs are not deducted.

### Capitalization Rate (Cap Rate)

**Definition:** The cap rate relates a property's annual Net Operating Income to its value or purchase price.

**Formula / structure:** Cap Rate = NOI / Property Value

**Investment use:** It provides a simple unlevered income-yield indicator and can also be used to capitalize stabilized NOI into an estimated value.

**Important caution:** Cap rates should be compared only with appropriate properties and markets and do not capture the full timing of future cash flows.

## Key Difference

**Gross rental yield** uses gross rent before operating expenses, while **Cap Rate** uses Net Operating Income (NOI).

Therefore, gross rental yield is a quick screening metric, whereas Cap Rate is generally a more refined income-property metric because operating expenses are incorporated through NOI.

## Investment Interpretation

A valuation metric should normally be interpreted together with market evidence, legal feasibility, property condition, risk and the investor's holding period rather than used as a standalone decision rule.

**Model limitation:** Area prices, yields, flood, soil and market indicators are screening-level inputs. They are not transaction-level appraisal evidence and should not replace property-specific legal, technical and financial due diligence.

**Disclaimer:** This chatbot is an educational investment-screening prototype and does not provide legal, tax, engineering, valuation or financial advice. Regulations, market conditions and transaction requirements may change. Users should verify current requirements and property-specific information with competent authorities, qualified legal advisers, banks, engineers, valuers and other professional advisers.


TEST 2: How do I calculate NPV for a real estate investment?
Detected route: valuation_calculation



# 🇹🇭 Bangkok Foreign Investor AI

## Valuation Calculation Guidance

### Net Present Value (NPV)

**Formula / structure:** NPV = Sum of discounted future cash flows - Initial Investment

To calculate NPV for a real-estate investment, you normally need:

- Initial investment / purchase price including relevant transaction costs
- Expected periodic net cash flows
- Holding period
- Expected sale or terminal proceeds
- Selling costs
- Discount rate / required rate of return

The simplified logic is:

**NPV = Present value of future net cash flows + present value of terminal proceeds - initial investment.**

If the modeled NPV is positive, the investment creates value relative to the selected discount rate under the stated assumptions. A negative NPV suggests the modeled return does not meet that required return.

## Inputs & Assumptions

A reliable property-specific calculation requires actual property inputs. The chatbot should not invent missing rents, costs, cap rates, resale values or discount rates.

## Model Limitation

Area prices, yields, flood, soil and market indicators are screening-level inputs. They are not transaction-level appraisal evidence and should not replace property-specific legal, technical and financial due diligence.

## Disclaimer

This chatbot is an educational investment-screening prototype and does not provide legal, tax, engineering, valuation or financial advice. Regulations, market conditions and transaction requirements may change. Users should verify current requirements and property-specific information with competent authorities, qualified legal advisers, banks, engineers, valuers and other professional advisers.

In [11]:
# CELL 7B - Legal & Regulatory Response Engine


# ============================================================
# 1. LAND TITLE DETECTION HELPERS
# ============================================================

LAND_TITLE_TERMS = {
    "chanote": [
        "chanote",
        "n.s.4",
        "ns4",
        "nor sor 4"
    ],

    "nor_sor_3_khor": [
        "nor sor 3 khor",
        "nor sor 3 kor",
        "n.s.3k",
        "ns3k",
        "nor sor 3 ghor"
    ],

    "nor_sor_3": [
        "nor sor 3",
        "n.s.3",
        "ns3"
    ],

    "sor_por_gor": [
        "sor por gor",
        "sor por ghor",
        "spk",
        "agricultural land reform"
    ]
}


def detect_land_titles_from_query(query):
    query = normalize_text(query)

    detected = []

    # Longer terms first
    title_items = []

    for title_key, terms in LAND_TITLE_TERMS.items():
        for term in terms:
            title_items.append(
                (normalize_text(term), title_key)
            )

    title_items = sorted(
        title_items,
        key=lambda x: len(x[0]),
        reverse=True
    )

    for term, title_key in title_items:

        pattern = (
            r"(?<!\w)"
            + re.escape(term)
            + r"(?!\w)"
        )

        if re.search(pattern, query):

            if title_key not in detected:
                detected.append(title_key)

    return detected


# ============================================================
# 2. LAND TITLE EXPLANATION
# ============================================================

def format_land_title_block(title_key):

    data = THAI_LAND_TITLES.get(title_key)

    if data is None:
        return (
            "The requested Thai land-title category is not "
            "available in the structured knowledge base."
        )

    lines = []

    lines.append(
        f"### {data['formal_name']}"
    )

    lines.append("")

    lines.append(
        f"**Ownership / legal quality:** "
        f"{data['ownership_quality']}"
    )

    lines.append("")

    lines.append(
        f"**Boundary-risk screening:** "
        f"{data['boundary_risk']}"
    )

    lines.append("")

    lines.append(
        f"**Survey basis:** "
        f"{data['survey_basis']}"
    )

    lines.append("")

    lines.append(
        f"**Transferability:** "
        f"{data['transferability']}"
    )

    lines.append("")

    lines.append(
        f"**Investment screening view:** "
        f"{data['screening_view']}"
    )

    return "\n".join(lines)


# ============================================================
# 3. CORE LEGAL RULE SELECTOR
# ============================================================

def select_legal_framework(parsed):

    property_types = parsed.get(
        "property_types",
        []
    )

    structures = parsed.get(
        "purchase_structures",
        []
    )

    # Land first: strongest regulatory issue
    if "land" in property_types:

        if "leasehold" in structures:
            return "leasehold"

        return "foreign_land_freehold"

    # Condominium
    if "condominium" in property_types:
        return "foreign_condominium_freehold"

    # House / townhouse / villa category
    if "house" in property_types:

        if "leasehold" in structures:
            return "leasehold"

        return "foreign_house"

    # Explicit leasehold without property type
    if "leasehold" in structures:
        return "leasehold"

    return None


# ============================================================
# 4. MAIN LEGAL RESPONSE
# ============================================================

def build_legal_response(parsed):
    """
    Produce regulation-first legal screening for foreign-investor
    property questions.

    The response is educational screening only and does not replace
    transaction-specific Thai legal advice.
    """

    query = parsed.get(
        "normalized_query",
        ""
    )

    property_types = parsed.get(
        "property_types",
        []
    )

    structures = parsed.get(
        "purchase_structures",
        []
    )

    locations = parsed.get(
        "locations",
        []
    )

    detected_titles = detect_land_titles_from_query(
        query
    )

    response = []

    response.append(
        "# 🇹🇭 Bangkok Foreign Investor AI"
    )

    response.append("")

    response.append(
        "## Regulation-First Legal Screening"
    )

    response.append("")


    # ========================================================
    # A. LAND TITLE QUESTIONS
    # ========================================================

    if detected_titles:

        response.append(
            "The question refers to Thai land-title documentation. "
            "The title category is important because ownership quality, "
            "survey certainty, transferability and boundary risk differ."
        )

        response.append("")

        for title_key in detected_titles:

            response.append(
                format_land_title_block(
                    title_key
                )
            )

            response.append("")

        if len(detected_titles) >= 2:

            response.append(
                "## Comparison Interpretation"
            )

            response.append("")

            response.append(
                "These title categories should not be treated as legally "
                "equivalent. A stronger title and more precise boundary "
                "documentation generally reduce title and boundary uncertainty, "
                "but the specific document and current Land Office records "
                "must still be verified."
            )

        response.append("")

        response.append(
            "## Due Diligence"
        )

        response.append("")

        response.append(
            format_bullet_list([
                "Inspect the original land-title document.",
                "Verify the title and current ownership with the competent Land Office.",
                "Check boundaries, access and rights-of-way.",
                "Check registered encumbrances or restrictions.",
                "Confirm whether the title can legally support the intended transaction and use."
            ])
        )

        response.append("")

        response.append(
            "## Disclaimer"
        )

        response.append("")

        response.append(
            LEGAL_DISCLAIMER
        )

        return "\n".join(response)


    # ========================================================
    # B. ZONING / DEVELOPMENT / PERMITS
    # ========================================================

    development_terms = [
        "zoning",
        "zone",
        "building permit",
        "permit",
        "planning",
        "urban planning",
        "eia",
        "environmental impact",
        "height restriction",
        "development regulation",
        "construction permit"
    ]

    if any(
        term in query
        for term in development_terms
    ):

        response.append(
            "The question concerns **property-development regulation "
            "and land-use compliance**."
        )

        response.append("")

        response.append(
            "A property or development should be screened for:"
        )

        response.append("")

        response.append(
            format_bullet_list(
                DEVELOPMENT_REGULATION_CHECKLIST
            )
        )

        response.append("")

        response.append(
            "## Investment Interpretation"
        )

        response.append("")

        response.append(
            "Financial attractiveness should not be treated as decisive "
            "until the intended property use is compatible with zoning, "
            "building-control requirements and required approvals."
        )

        response.append("")

        response.append(
            "The applicable requirements can depend on the exact site, "
            "building type, scale and intended use."
        )

        response.append("")

        response.append(
            "## Disclaimer"
        )

        response.append("")

        response.append(
            LEGAL_DISCLAIMER
        )

        return "\n".join(response)


    # ========================================================
    # C. OFF-PLAN / CONSUMER PROTECTION
    # ========================================================

    consumer_terms = [
        "off-plan",
        "off plan",
        "pre-construction",
        "pre construction",
        "developer contract",
        "developer agreement",
        "buying from developer",
        "consumer protection"
    ]

    if any(
        term in query
        for term in consumer_terms
    ):

        response.append(
            "The question concerns **off-plan / developer-related "
            "consumer protection and transaction due diligence**."
        )

        response.append("")

        response.append(
            "Important items to review include:"
        )

        response.append("")

        response.append(
            format_bullet_list(
                CONSUMER_PROTECTION_CHECKLIST
            )
        )

        response.append("")

        response.append(
            "## Investor Caution"
        )

        response.append("")

        response.append(
            "Marketing material, show units and projected completion "
            "standards should be checked against the written contract "
            "and legally binding project documentation."
        )

        response.append("")

        response.append(
            "## Disclaimer"
        )

        response.append("")

        response.append(
            LEGAL_DISCLAIMER
        )

        return "\n".join(response)


    # ========================================================
    # D. CORE FOREIGN-OWNERSHIP FRAMEWORK
    # ========================================================

    framework_key = select_legal_framework(
        parsed
    )

    if framework_key is None:

        response.append(
            "The question contains a legal or ownership signal, "
            "but the property type or intended ownership structure "
            "is not sufficiently clear."
        )

        response.append("")

        response.append(
            "For a more reliable legal screening, specify whether "
            "the investment involves a **condominium, house, land, "
            "commercial property or another property type**, and "
            "whether **freehold or leasehold** is intended."
        )

        response.append("")

        response.append(
            "## General Legal Due Diligence"
        )

        response.append("")

        response.append(
            format_bullet_list(
                DUE_DILIGENCE_CATEGORIES["legal"]
            )
        )

        response.append("")

        response.append(
            "## Disclaimer"
        )

        response.append("")

        response.append(
            LEGAL_DISCLAIMER
        )

        return "\n".join(response)


    legal_data = LEGAL_RULES[
        framework_key
    ]

    # Property / structure summary
    if property_types:

        property_label = ", ".join(
            p.replace("_", " ").title()
            for p in property_types
        )

    else:

        property_label = "Not clearly specified"

    if structures:

        structure_label = ", ".join(
            s.title()
            for s in structures
        )

    else:

        if framework_key == "foreign_condominium_freehold":
            structure_label = "Freehold screening assumed"

        elif framework_key == "foreign_land_freehold":
            structure_label = "Direct ownership / freehold screening"

        else:
            structure_label = "Not clearly specified"


    response.append(
        f"**Property type:** {property_label}"
    )

    response.append("")

    response.append(
        f"**Ownership structure:** {structure_label}"
    )

    response.append("")

    response.append(
        f"**Regulatory status:** {legal_data['status']}"
    )

    response.append("")

    response.append(
        f"**Core rule:** {legal_data['rule']}"
    )

    response.append("")

    response.append(
        f"**Key condition:** {legal_data['key_condition']}"
    )

    response.append("")

    response.append(
        "## Required Legal Due Diligence"
    )

    response.append("")

    response.append(
        format_bullet_list(
            legal_data["due_diligence"]
        )
    )


    # ========================================================
    # E. CONDO-SPECIFIC FOREIGN OWNERSHIP NOTE
    # ========================================================

    if framework_key == "foreign_condominium_freehold":

        response.append("")

        response.append(
            "## Foreign Condominium Ownership"
        )

        response.append("")

        response.append(
            "Foreign condominium ownership is generally subject to "
            "the statutory foreign-ownership quota. For screening "
            "purposes, this is commonly described as a limit of "
            "approximately **49% of the total unit area of the "
            "condominium** being foreign-owned."
        )

        response.append("")

        response.append(
            "The available foreign quota must therefore be verified "
            "for the specific condominium before transfer."
        )


    # ========================================================
    # F. LAND-SPECIFIC LEGAL GATE
    # ========================================================

    if framework_key == "foreign_land_freehold":

        response.append("")

        response.append(
            "## Legal Gate"
        )

        response.append("")

        response.append(
            "**The chatbot should not proceed directly to an investment "
            "ranking for ordinary direct foreign freehold land ownership.**"
        )

        response.append("")

        response.append(
            "The legally feasible ownership or investment structure "
            "should be clarified first."
        )

        response.append("")

        response.append(
            "Potential alternatives may include professionally reviewed "
            "leasehold or other lawful structures, depending on the "
            "specific transaction."
        )


    # ========================================================
    # G. HOUSE-SPECIFIC NOTE
    # ========================================================

    if framework_key == "foreign_house":

        response.append("")

        response.append(
            "## Building vs. Land"
        )

        response.append("")

        response.append(
            "For foreign investors, ownership of a building and ownership "
            "of the land beneath it should be analysed separately."
        )

        response.append("")

        response.append(
            "A legally feasible structure may therefore involve different "
            "rights in the building and the underlying land."
        )


    # ========================================================
    # H. LEASEHOLD NOTE
    # ========================================================

    if framework_key == "leasehold":

        response.append("")

        response.append(
            "## Leasehold Interpretation"
        )

        response.append("")

        response.append(
            "Leasehold provides contractual use rights rather than "
            "ordinary freehold ownership of the underlying land."
        )

        response.append("")

        response.append(
            "The lease term, registration, renewal wording, assignment "
            "rights and termination clauses should therefore be reviewed "
            "carefully."
        )


    # ========================================================
    # I. LOCATION CONTEXT
    # ========================================================

    if locations:

        response.append("")

        response.append(
            "## Location Context"
        )

        response.append("")

        response.append(
            "The following Bangkok area was detected:"
            if len(locations) == 1
            else "The following Bangkok areas were detected:"
        )

        response.append("")

        for location in locations:
            response.append(
                f"- {location}"
            )

        response.append("")

        response.append(
            "Location attractiveness should only be evaluated after "
            "the ownership structure is legally feasible."
        )


    response.append("")

    response.append(
        "## Model Limitation"
    )

    response.append("")

    response.append(
        "This is regulation-first investment screening. It does not "
        "determine the legal validity of a specific contract, title, "
        "ownership structure or transaction."
    )

    response.append("")

    response.append(
        "## Disclaimer"
    )

    response.append("")

    response.append(
        LEGAL_DISCLAIMER
    )

    return "\n".join(response)


print("✓ Legal & regulatory response engine loaded successfully")

✓ Legal & regulatory response engine loaded successfully


In [12]:
# CELL 7B TEST - Legal Response Quality

legal_test_questions = [
    "Can a foreigner buy land in Bangkok?",
    "Can a German buy a condominium in Bangkok?",
    "I want a leasehold house in Bangkok. What should I know?",
    "What is a Chanote and how is it different from Nor Sor 3?"
]

for i, question in enumerate(legal_test_questions, 1):
    print("\n" + "=" * 90)
    print(f"TEST {i}: {question}")
    print("=" * 90)

    parsed = parse_investment_query(question)
    route = route_investment_query(parsed)

    print(f"Detected route: {route['primary_route']}")
    print()

    answer = build_legal_response(parsed)

    display(Markdown(answer))


TEST 1: Can a foreigner buy land in Bangkok?
Detected route: legal_restriction



# 🇹🇭 Bangkok Foreign Investor AI

## Regulation-First Legal Screening

**Property type:** Land

**Ownership structure:** Direct ownership / freehold screening

**Regulatory status:** GENERALLY RESTRICTED

**Core rule:** Foreign individuals generally cannot directly own land in Thailand under ordinary circumstances, subject to limited statutory exceptions.

**Key condition:** A foreign investor should not assume ordinary direct freehold land ownership is legally available.

## Required Legal Due Diligence

- Obtain qualified Thai legal advice before committing funds.
- Verify the land title at the competent Land Office.
- Confirm the exact permitted ownership or investment structure.
- Do not use nominee arrangements to circumvent Thai law.
- Review zoning, access, easements and development restrictions.

## Legal Gate

**The chatbot should not proceed directly to an investment ranking for ordinary direct foreign freehold land ownership.**

The legally feasible ownership or investment structure should be clarified first.

Potential alternatives may include professionally reviewed leasehold or other lawful structures, depending on the specific transaction.

## Model Limitation

This is regulation-first investment screening. It does not determine the legal validity of a specific contract, title, ownership structure or transaction.

## Disclaimer

This chatbot is an educational investment-screening prototype and does not provide legal, tax, engineering, valuation or financial advice. Regulations, market conditions and transaction requirements may change. Users should verify current requirements and property-specific information with competent authorities, qualified legal advisers, banks, engineers, valuers and other professional advisers.


TEST 2: Can a German buy a condominium in Bangkok?
Detected route: property_screening



# 🇹🇭 Bangkok Foreign Investor AI

## Regulation-First Legal Screening

**Property type:** Condominium

**Ownership structure:** Freehold screening assumed

**Regulatory status:** GENERALLY PERMITTED

**Core rule:** Foreign individuals may generally own qualifying condominium units in Thailand in their own name, subject to applicable statutory conditions and the foreign ownership quota.

**Key condition:** Foreign ownership must generally remain within the applicable condominium foreign-ownership quota.

## Required Legal Due Diligence

- Verify that the condominium is legally registered.
- Confirm the remaining foreign ownership quota with the condominium juristic person.
- Verify the unit title and ownership records with the competent Land Office.
- Confirm foreign-fund remittance and transfer-document requirements before completion.
- Review common-area fees, sinking fund obligations and building financial statements.

## Foreign Condominium Ownership

Foreign condominium ownership is generally subject to the statutory foreign-ownership quota. For screening purposes, this is commonly described as a limit of approximately **49% of the total unit area of the condominium** being foreign-owned.

The available foreign quota must therefore be verified for the specific condominium before transfer.

## Model Limitation

This is regulation-first investment screening. It does not determine the legal validity of a specific contract, title, ownership structure or transaction.

## Disclaimer

This chatbot is an educational investment-screening prototype and does not provide legal, tax, engineering, valuation or financial advice. Regulations, market conditions and transaction requirements may change. Users should verify current requirements and property-specific information with competent authorities, qualified legal advisers, banks, engineers, valuers and other professional advisers.


TEST 3: I want a leasehold house in Bangkok. What should I know?
Detected route: legal_guidance



# 🇹🇭 Bangkok Foreign Investor AI

## Regulation-First Legal Screening

**Property type:** House

**Ownership structure:** Leasehold

**Regulatory status:** POTENTIALLY AVAILABLE

**Core rule:** Leasehold may provide contractual use rights without transferring freehold ownership of the underlying land.

**Key condition:** Lease term, registration, renewal language, transfer rights and termination provisions require contract-specific legal review.

## Required Legal Due Diligence

- Verify the legal owner and title of the leased property.
- Check whether the lease must be registered.
- Review lease term and renewal clauses carefully.
- Review assignment, inheritance and termination provisions.
- Do not treat contractual renewal expectations as guaranteed ownership rights.

## Leasehold Interpretation

Leasehold provides contractual use rights rather than ordinary freehold ownership of the underlying land.

The lease term, registration, renewal wording, assignment rights and termination clauses should therefore be reviewed carefully.

## Model Limitation

This is regulation-first investment screening. It does not determine the legal validity of a specific contract, title, ownership structure or transaction.

## Disclaimer

This chatbot is an educational investment-screening prototype and does not provide legal, tax, engineering, valuation or financial advice. Regulations, market conditions and transaction requirements may change. Users should verify current requirements and property-specific information with competent authorities, qualified legal advisers, banks, engineers, valuers and other professional advisers.


TEST 4: What is a Chanote and how is it different from Nor Sor 3?
Detected route: legal_guidance



# 🇹🇭 Bangkok Foreign Investor AI

## Regulation-First Legal Screening

The question refers to Thai land-title documentation. The title category is important because ownership quality, survey certainty, transferability and boundary risk differ.

### Nor Sor 3 (N.S.3)

**Ownership / legal quality:** Confirmed claim / use certificate

**Boundary-risk screening:** Higher relative risk

**Survey basis:** Less precise boundary definition

**Transferability:** May be transferable subject to legal requirements

**Investment screening view:** Boundary and legal due diligence is particularly important.

### Chanote (N.S.4)

**Ownership / legal quality:** Full title deed

**Boundary-risk screening:** Very low relative risk

**Survey basis:** Formal surveyed boundaries / boundary markers

**Transferability:** Generally transferable, subject to applicable law

**Investment screening view:** Usually the strongest land-title form for ownership verification, but the specific deed and Land Office records must still be checked.

## Comparison Interpretation

These title categories should not be treated as legally equivalent. A stronger title and more precise boundary documentation generally reduce title and boundary uncertainty, but the specific document and current Land Office records must still be verified.

## Due Diligence

- Inspect the original land-title document.
- Verify the title and current ownership with the competent Land Office.
- Check boundaries, access and rights-of-way.
- Check registered encumbrances or restrictions.
- Confirm whether the title can legally support the intended transaction and use.

## Disclaimer

This chatbot is an educational investment-screening prototype and does not provide legal, tax, engineering, valuation or financial advice. Regulations, market conditions and transaction requirements may change. Users should verify current requirements and property-specific information with competent authorities, qualified legal advisers, banks, engineers, valuers and other professional advisers.

In [13]:
# CELL 7C - Technical Due-Diligence Response Engine


def build_technical_response(parsed):
    """
    Build technical due-diligence guidance for Bangkok real-estate
    investment questions.

    The engine distinguishes between:
    1. general technical-risk questions,
    2. location-specific technical screening,
    3. property-specific due diligence.

    It must not invent building- or site-specific engineering facts.
    """

    query = parsed.get("normalized_query", "")
    locations = parsed.get("locations", [])
    property_types = parsed.get("property_types", [])
    technical_risks = parsed.get("technical_risks", [])

    response = []

    response.append("# 🇹🇭 Bangkok Foreign Investor AI")
    response.append("")
    response.append("## Technical Due-Diligence Screening")
    response.append("")

    # ========================================================
    # 1. DETERMINE TECHNICAL TOPICS
    # ========================================================

    wants_flood = (
        "flood" in technical_risks
        or "flooding" in query
        or "drainage" in query
        or "waterlogging" in query
    )

    wants_soil = (
        "soil" in technical_risks
        or "geotechnical" in query
        or "ground condition" in query
        or "subsidence" in query
        or "settlement" in query
    )

    wants_structure = (
        "structure" in technical_risks
        or "structural" in query
        or "foundation" in query
        or "building condition" in query
        or "crack" in query
    )

    # If the router sent the question here but no specific
    # technical subtopic was detected, provide broad screening.
    if not any([
        wants_flood,
        wants_soil,
        wants_structure
    ]):
        wants_flood = True
        wants_soil = True
        wants_structure = True


    # ========================================================
    # 2. CONTEXT SUMMARY
    # ========================================================

    if locations:
        response.append(
            "**Location detected:** "
            + ", ".join(locations)
        )
        response.append("")

    if property_types:
        response.append(
            "**Property type detected:** "
            + ", ".join(
                item.replace("_", " ").title()
                for item in property_types
            )
        )
        response.append("")

    response.append(
        "Technical risk should be assessed at both **area level** "
        "and **property/site level**. Area-level screening can identify "
        "issues that deserve attention, but it cannot determine the "
        "condition or engineering suitability of an individual property."
    )


    # ========================================================
    # 3. FLOOD / DRAINAGE SCREENING
    # ========================================================

    if wants_flood:

        response.append("")
        response.append("## 1. Flood & Drainage Risk")
        response.append("")

        response.append(
            "Flood exposure should not be assessed only at district level. "
            "Street, soi, plot and building access conditions can differ "
            "materially within the same Bangkok area."
        )

        response.append("")

        response.append("### What to Check")
        response.append("")

        response.append(
            format_bullet_list([
                "Historical flood or waterlogging at the specific street, soi and building.",
                "Drainage performance during heavy rainfall.",
                "Elevation of the plot and building entrance relative to the surrounding road.",
                "Basement, parking and ground-floor exposure to water intrusion.",
                "Access to the property during severe rainfall.",
                "Building flood-protection measures such as barriers, pumps and drainage systems.",
                "Evidence of previous water damage, dampness or repeated repairs.",
                "Local infrastructure and drainage improvements that may affect future exposure."
            ])
        )

        response.append("")
        response.append("### Investment Interpretation")
        response.append("")

        response.append(
            "Flood risk can affect repair costs, tenant demand, accessibility, "
            "insurance considerations, resale liquidity and the reliability "
            "of expected investment cash flows."
        )


    # ========================================================
    # 4. SOIL / GEOTECHNICAL SCREENING
    # ========================================================

    if wants_soil:

        response.append("")
        response.append("## 2. Soil & Geotechnical Conditions")
        response.append("")

        response.append(
            "Bangkok-area soil information is useful as a screening signal, "
            "but the suitability of a particular development depends on "
            "site-specific geotechnical and foundation conditions."
        )

        response.append("")

        response.append("### What to Check")
        response.append("")

        response.append(
            format_bullet_list([
                "Available geotechnical or soil-investigation reports for the project.",
                "Foundation type and depth.",
                "Evidence of differential settlement or subsidence.",
                "Cracking, tilting or deformation that may indicate movement.",
                "Groundwater and drainage conditions where relevant.",
                "Engineering design assumptions for the specific building.",
                "Major repair history related to foundations or structural movement.",
                "Independent engineering review where material uncertainty exists."
            ])
        )

        response.append("")
        response.append("### Investment Interpretation")
        response.append("")

        response.append(
            "Soil and foundation issues can create significant capital expenditure, "
            "maintenance and safety implications. A favorable area-level investment "
            "score should therefore never override material engineering concerns."
        )


    # ========================================================
    # 5. STRUCTURAL / BUILDING CONDITION
    # ========================================================

    if wants_structure:

        response.append("")
        response.append("## 3. Building & Structural Condition")
        response.append("")

        response.append(
            "For an existing property, technical due diligence should also "
            "consider the physical condition and maintenance history of the building."
        )

        response.append("")

        response.append("### What to Check")
        response.append("")

        response.append(
            format_bullet_list([
                "Visible structural cracks, deformation or signs of settlement.",
                "Water intrusion, roof leakage and façade deterioration.",
                "Condition of mechanical, electrical and plumbing systems.",
                "Elevator and fire-safety system condition where applicable.",
                "Building maintenance history and major planned repairs.",
                "Common-area condition for condominium projects.",
                "Evidence of deferred maintenance.",
                "Independent building inspection where appropriate."
            ])
        )


    # ========================================================
    # 6. LOCATION-SPECIFIC SCREENING
    # ========================================================

    if locations:

        response.append("")
        response.append("## Location-Specific Screening")
        response.append("")

        for location in locations:

            zone_data = BANGKOK_ZONES.get(location)

            if zone_data is None:
                response.append(
                    f"### {location}"
                )
                response.append("")
                response.append(
                    "The location was detected, but no structured technical "
                    "screening record is available. The chatbot will not invent "
                    "location-specific flood or soil information."
                )
                response.append("")
                continue

            response.append(
                f"### {location}"
            )
            response.append("")

            flood_note = zone_data.get("flood")
            soil_note = zone_data.get("soil")

            if wants_flood:
                if flood_note:
                    response.append(
                        f"**Area-level flood / drainage screening:** {flood_note}"
                    )
                else:
                    response.append(
                        "**Area-level flood / drainage screening:** "
                        "No specific structured note is available."
                    )

                response.append("")

            if wants_soil:
                if soil_note:
                    response.append(
                        f"**Area-level soil / engineering screening:** {soil_note}"
                    )
                else:
                    response.append(
                        "**Area-level soil / engineering screening:** "
                        "No specific structured note is available."
                    )

                response.append("")

        response.append(
            "**Important:** These are area-level screening indicators only. "
            "They do not establish the flood history, soil quality, foundation "
            "condition or structural safety of a specific property."
        )


    # ========================================================
    # 7. INVESTOR DUE-DILIGENCE PRIORITIES
    # ========================================================

    response.append("")
    response.append("## Recommended Technical Due Diligence")
    response.append("")

    recommended_checks = []

    if wants_flood:
        recommended_checks.extend([
            "Inspect the property and surrounding access after or during heavy rainfall where practicable.",
            "Ask the seller, developer, juristic person or building management about historical flooding and water intrusion."
        ])

    if wants_soil:
        recommended_checks.extend([
            "Request available soil, foundation and geotechnical documentation.",
            "Use a qualified engineer where foundation or settlement concerns exist."
        ])

    if wants_structure:
        recommended_checks.extend([
            "Review building maintenance and major-repair records.",
            "Consider an independent technical inspection before committing funds."
        ])

    recommended_checks.extend([
        "Do not rely solely on area averages or neighborhood reputation.",
        "Reflect material technical risks in the investment decision, expected costs and required return."
    ])

    # Remove duplicates while preserving order
    recommended_checks = list(
        dict.fromkeys(recommended_checks)
    )

    response.append(
        format_bullet_list(
            recommended_checks
        )
    )


    # ========================================================
    # 8. MODEL LIMITATION
    # ========================================================

    response.append("")
    response.append("## Model Limitation")
    response.append("")

    response.append(
        DATA_LIMITATION_NOTE
    )

    response.append("")

    response.append(
        "The chatbot does not perform a physical inspection, engineering "
        "assessment, hydrological study or geotechnical investigation."
    )


    # ========================================================
    # 9. DISCLAIMER
    # ========================================================

    response.append("")
    response.append("## Disclaimer")
    response.append("")

    response.append(
        LEGAL_DISCLAIMER
    )

    return "\n".join(response)


print("✓ Technical due-diligence response engine loaded successfully")

✓ Technical due-diligence response engine loaded successfully


In [14]:
# CELL 7D - Technical Due-Diligence Stress Test

technical_test_questions = [
    "What should I check regarding flood and soil conditions?",
    "What flood risks should I consider when buying a condo in On Nut?",
    "I am considering a property in Bang Kapi. What should I know about soil and foundation risks?",
    "What structural problems should I check before buying an older condominium in Bangkok?",
    "I want to buy a condo in On Nut. What should I check regarding flooding, soil conditions and the building structure?"
]

for i, question in enumerate(technical_test_questions, 1):

    print("\n" + "=" * 90)
    print(f"TEST {i}: {question}")
    print("=" * 90)

    parsed = parse_investment_query(question)
    route = route_investment_query(parsed)

    print(f"Detected route: {route['primary_route']}")
    print(f"Locations: {parsed.get('locations', [])}")
    print(f"Technical risks: {parsed.get('technical_risks', [])}")
    print()

    if route["primary_route"] == "technical_due_diligence":
        answer = build_technical_response(parsed)
        display(Markdown(answer))
    else:
        print(
            f"⚠ Unexpected route: {route['primary_route']} "
            f"(expected technical_due_diligence)"
        )


TEST 1: What should I check regarding flood and soil conditions?
Detected route: technical_due_diligence
Locations: []
Technical risks: ['flood', 'soil']



# 🇹🇭 Bangkok Foreign Investor AI

## Technical Due-Diligence Screening

Technical risk should be assessed at both **area level** and **property/site level**. Area-level screening can identify issues that deserve attention, but it cannot determine the condition or engineering suitability of an individual property.

## 1. Flood & Drainage Risk

Flood exposure should not be assessed only at district level. Street, soi, plot and building access conditions can differ materially within the same Bangkok area.

### What to Check

- Historical flood or waterlogging at the specific street, soi and building.
- Drainage performance during heavy rainfall.
- Elevation of the plot and building entrance relative to the surrounding road.
- Basement, parking and ground-floor exposure to water intrusion.
- Access to the property during severe rainfall.
- Building flood-protection measures such as barriers, pumps and drainage systems.
- Evidence of previous water damage, dampness or repeated repairs.
- Local infrastructure and drainage improvements that may affect future exposure.

### Investment Interpretation

Flood risk can affect repair costs, tenant demand, accessibility, insurance considerations, resale liquidity and the reliability of expected investment cash flows.

## 2. Soil & Geotechnical Conditions

Bangkok-area soil information is useful as a screening signal, but the suitability of a particular development depends on site-specific geotechnical and foundation conditions.

### What to Check

- Available geotechnical or soil-investigation reports for the project.
- Foundation type and depth.
- Evidence of differential settlement or subsidence.
- Cracking, tilting or deformation that may indicate movement.
- Groundwater and drainage conditions where relevant.
- Engineering design assumptions for the specific building.
- Major repair history related to foundations or structural movement.
- Independent engineering review where material uncertainty exists.

### Investment Interpretation

Soil and foundation issues can create significant capital expenditure, maintenance and safety implications. A favorable area-level investment score should therefore never override material engineering concerns.

## Recommended Technical Due Diligence

- Inspect the property and surrounding access after or during heavy rainfall where practicable.
- Ask the seller, developer, juristic person or building management about historical flooding and water intrusion.
- Request available soil, foundation and geotechnical documentation.
- Use a qualified engineer where foundation or settlement concerns exist.
- Do not rely solely on area averages or neighborhood reputation.
- Reflect material technical risks in the investment decision, expected costs and required return.

## Model Limitation

Area prices, yields, flood, soil and market indicators are screening-level inputs. They are not transaction-level appraisal evidence and should not replace property-specific legal, technical and financial due diligence.

The chatbot does not perform a physical inspection, engineering assessment, hydrological study or geotechnical investigation.

## Disclaimer

This chatbot is an educational investment-screening prototype and does not provide legal, tax, engineering, valuation or financial advice. Regulations, market conditions and transaction requirements may change. Users should verify current requirements and property-specific information with competent authorities, qualified legal advisers, banks, engineers, valuers and other professional advisers.


TEST 2: What flood risks should I consider when buying a condo in On Nut?
Detected route: technical_due_diligence
Locations: ['On Nut - Phra Khanong']
Technical risks: ['flood']



# 🇹🇭 Bangkok Foreign Investor AI

## Technical Due-Diligence Screening

**Location detected:** On Nut - Phra Khanong

**Property type detected:** Condominium

Technical risk should be assessed at both **area level** and **property/site level**. Area-level screening can identify issues that deserve attention, but it cannot determine the condition or engineering suitability of an individual property.

## 1. Flood & Drainage Risk

Flood exposure should not be assessed only at district level. Street, soi, plot and building access conditions can differ materially within the same Bangkok area.

### What to Check

- Historical flood or waterlogging at the specific street, soi and building.
- Drainage performance during heavy rainfall.
- Elevation of the plot and building entrance relative to the surrounding road.
- Basement, parking and ground-floor exposure to water intrusion.
- Access to the property during severe rainfall.
- Building flood-protection measures such as barriers, pumps and drainage systems.
- Evidence of previous water damage, dampness or repeated repairs.
- Local infrastructure and drainage improvements that may affect future exposure.

### Investment Interpretation

Flood risk can affect repair costs, tenant demand, accessibility, insurance considerations, resale liquidity and the reliability of expected investment cash flows.

## Location-Specific Screening

### On Nut - Phra Khanong

**Area-level flood / drainage screening:** Check soi-level drainage and access during heavy rainfall

**Important:** These are area-level screening indicators only. They do not establish the flood history, soil quality, foundation condition or structural safety of a specific property.

## Recommended Technical Due Diligence

- Inspect the property and surrounding access after or during heavy rainfall where practicable.
- Ask the seller, developer, juristic person or building management about historical flooding and water intrusion.
- Do not rely solely on area averages or neighborhood reputation.
- Reflect material technical risks in the investment decision, expected costs and required return.

## Model Limitation

Area prices, yields, flood, soil and market indicators are screening-level inputs. They are not transaction-level appraisal evidence and should not replace property-specific legal, technical and financial due diligence.

The chatbot does not perform a physical inspection, engineering assessment, hydrological study or geotechnical investigation.

## Disclaimer

This chatbot is an educational investment-screening prototype and does not provide legal, tax, engineering, valuation or financial advice. Regulations, market conditions and transaction requirements may change. Users should verify current requirements and property-specific information with competent authorities, qualified legal advisers, banks, engineers, valuers and other professional advisers.


TEST 3: I am considering a property in Bang Kapi. What should I know about soil and foundation risks?
Detected route: technical_due_diligence
Locations: ['Ramkhamhaeng - Bang Kapi']
Technical risks: ['soil']



# 🇹🇭 Bangkok Foreign Investor AI

## Technical Due-Diligence Screening

**Location detected:** Ramkhamhaeng - Bang Kapi

Technical risk should be assessed at both **area level** and **property/site level**. Area-level screening can identify issues that deserve attention, but it cannot determine the condition or engineering suitability of an individual property.

## 2. Soil & Geotechnical Conditions

Bangkok-area soil information is useful as a screening signal, but the suitability of a particular development depends on site-specific geotechnical and foundation conditions.

### What to Check

- Available geotechnical or soil-investigation reports for the project.
- Foundation type and depth.
- Evidence of differential settlement or subsidence.
- Cracking, tilting or deformation that may indicate movement.
- Groundwater and drainage conditions where relevant.
- Engineering design assumptions for the specific building.
- Major repair history related to foundations or structural movement.
- Independent engineering review where material uncertainty exists.

### Investment Interpretation

Soil and foundation issues can create significant capital expenditure, maintenance and safety implications. A favorable area-level investment score should therefore never override material engineering concerns.

## 3. Building & Structural Condition

For an existing property, technical due diligence should also consider the physical condition and maintenance history of the building.

### What to Check

- Visible structural cracks, deformation or signs of settlement.
- Water intrusion, roof leakage and façade deterioration.
- Condition of mechanical, electrical and plumbing systems.
- Elevator and fire-safety system condition where applicable.
- Building maintenance history and major planned repairs.
- Common-area condition for condominium projects.
- Evidence of deferred maintenance.
- Independent building inspection where appropriate.

## Location-Specific Screening

### Ramkhamhaeng - Bang Kapi

**Area-level soil / engineering screening:** Soft clay; low-lying site conditions require engineering review

**Important:** These are area-level screening indicators only. They do not establish the flood history, soil quality, foundation condition or structural safety of a specific property.

## Recommended Technical Due Diligence

- Request available soil, foundation and geotechnical documentation.
- Use a qualified engineer where foundation or settlement concerns exist.
- Review building maintenance and major-repair records.
- Consider an independent technical inspection before committing funds.
- Do not rely solely on area averages or neighborhood reputation.
- Reflect material technical risks in the investment decision, expected costs and required return.

## Model Limitation

Area prices, yields, flood, soil and market indicators are screening-level inputs. They are not transaction-level appraisal evidence and should not replace property-specific legal, technical and financial due diligence.

The chatbot does not perform a physical inspection, engineering assessment, hydrological study or geotechnical investigation.

## Disclaimer

This chatbot is an educational investment-screening prototype and does not provide legal, tax, engineering, valuation or financial advice. Regulations, market conditions and transaction requirements may change. Users should verify current requirements and property-specific information with competent authorities, qualified legal advisers, banks, engineers, valuers and other professional advisers.


TEST 4: What structural problems should I check before buying an older condominium in Bangkok?
Detected route: technical_due_diligence
Locations: []
Technical risks: ['soil']



# 🇹🇭 Bangkok Foreign Investor AI

## Technical Due-Diligence Screening

**Property type detected:** Condominium

Technical risk should be assessed at both **area level** and **property/site level**. Area-level screening can identify issues that deserve attention, but it cannot determine the condition or engineering suitability of an individual property.

## 2. Soil & Geotechnical Conditions

Bangkok-area soil information is useful as a screening signal, but the suitability of a particular development depends on site-specific geotechnical and foundation conditions.

### What to Check

- Available geotechnical or soil-investigation reports for the project.
- Foundation type and depth.
- Evidence of differential settlement or subsidence.
- Cracking, tilting or deformation that may indicate movement.
- Groundwater and drainage conditions where relevant.
- Engineering design assumptions for the specific building.
- Major repair history related to foundations or structural movement.
- Independent engineering review where material uncertainty exists.

### Investment Interpretation

Soil and foundation issues can create significant capital expenditure, maintenance and safety implications. A favorable area-level investment score should therefore never override material engineering concerns.

## 3. Building & Structural Condition

For an existing property, technical due diligence should also consider the physical condition and maintenance history of the building.

### What to Check

- Visible structural cracks, deformation or signs of settlement.
- Water intrusion, roof leakage and façade deterioration.
- Condition of mechanical, electrical and plumbing systems.
- Elevator and fire-safety system condition where applicable.
- Building maintenance history and major planned repairs.
- Common-area condition for condominium projects.
- Evidence of deferred maintenance.
- Independent building inspection where appropriate.

## Recommended Technical Due Diligence

- Request available soil, foundation and geotechnical documentation.
- Use a qualified engineer where foundation or settlement concerns exist.
- Review building maintenance and major-repair records.
- Consider an independent technical inspection before committing funds.
- Do not rely solely on area averages or neighborhood reputation.
- Reflect material technical risks in the investment decision, expected costs and required return.

## Model Limitation

Area prices, yields, flood, soil and market indicators are screening-level inputs. They are not transaction-level appraisal evidence and should not replace property-specific legal, technical and financial due diligence.

The chatbot does not perform a physical inspection, engineering assessment, hydrological study or geotechnical investigation.

## Disclaimer

This chatbot is an educational investment-screening prototype and does not provide legal, tax, engineering, valuation or financial advice. Regulations, market conditions and transaction requirements may change. Users should verify current requirements and property-specific information with competent authorities, qualified legal advisers, banks, engineers, valuers and other professional advisers.


TEST 5: I want to buy a condo in On Nut. What should I check regarding flooding, soil conditions and the building structure?
Detected route: technical_due_diligence
Locations: ['On Nut - Phra Khanong']
Technical risks: ['flood', 'soil']



# 🇹🇭 Bangkok Foreign Investor AI

## Technical Due-Diligence Screening

**Location detected:** On Nut - Phra Khanong

**Property type detected:** Condominium

Technical risk should be assessed at both **area level** and **property/site level**. Area-level screening can identify issues that deserve attention, but it cannot determine the condition or engineering suitability of an individual property.

## 1. Flood & Drainage Risk

Flood exposure should not be assessed only at district level. Street, soi, plot and building access conditions can differ materially within the same Bangkok area.

### What to Check

- Historical flood or waterlogging at the specific street, soi and building.
- Drainage performance during heavy rainfall.
- Elevation of the plot and building entrance relative to the surrounding road.
- Basement, parking and ground-floor exposure to water intrusion.
- Access to the property during severe rainfall.
- Building flood-protection measures such as barriers, pumps and drainage systems.
- Evidence of previous water damage, dampness or repeated repairs.
- Local infrastructure and drainage improvements that may affect future exposure.

### Investment Interpretation

Flood risk can affect repair costs, tenant demand, accessibility, insurance considerations, resale liquidity and the reliability of expected investment cash flows.

## 2. Soil & Geotechnical Conditions

Bangkok-area soil information is useful as a screening signal, but the suitability of a particular development depends on site-specific geotechnical and foundation conditions.

### What to Check

- Available geotechnical or soil-investigation reports for the project.
- Foundation type and depth.
- Evidence of differential settlement or subsidence.
- Cracking, tilting or deformation that may indicate movement.
- Groundwater and drainage conditions where relevant.
- Engineering design assumptions for the specific building.
- Major repair history related to foundations or structural movement.
- Independent engineering review where material uncertainty exists.

### Investment Interpretation

Soil and foundation issues can create significant capital expenditure, maintenance and safety implications. A favorable area-level investment score should therefore never override material engineering concerns.

## Location-Specific Screening

### On Nut - Phra Khanong

**Area-level flood / drainage screening:** Check soi-level drainage and access during heavy rainfall

**Area-level soil / engineering screening:** Bangkok soft-clay conditions

**Important:** These are area-level screening indicators only. They do not establish the flood history, soil quality, foundation condition or structural safety of a specific property.

## Recommended Technical Due Diligence

- Inspect the property and surrounding access after or during heavy rainfall where practicable.
- Ask the seller, developer, juristic person or building management about historical flooding and water intrusion.
- Request available soil, foundation and geotechnical documentation.
- Use a qualified engineer where foundation or settlement concerns exist.
- Do not rely solely on area averages or neighborhood reputation.
- Reflect material technical risks in the investment decision, expected costs and required return.

## Model Limitation

Area prices, yields, flood, soil and market indicators are screening-level inputs. They are not transaction-level appraisal evidence and should not replace property-specific legal, technical and financial due diligence.

The chatbot does not perform a physical inspection, engineering assessment, hydrological study or geotechnical investigation.

## Disclaimer

This chatbot is an educational investment-screening prototype and does not provide legal, tax, engineering, valuation or financial advice. Regulations, market conditions and transaction requirements may change. Users should verify current requirements and property-specific information with competent authorities, qualified legal advisers, banks, engineers, valuers and other professional advisers.

In [15]:
# CELL 8 - Bangkok Investment Scoring & Recommendation Engine


# ============================================================
# 1. STRATEGY WEIGHTS
# ============================================================

STRATEGY_WEIGHTS = {

    "balanced": {
        "yield": 0.20,
        "growth": 0.25,
        "transit": 0.20,
        "international_demand": 0.15,
        "affordability": 0.20
    },

    "income": {
        "yield": 0.35,
        "growth": 0.10,
        "transit": 0.15,
        "international_demand": 0.15,
        "affordability": 0.25
    },

    "growth": {
        "yield": 0.10,
        "growth": 0.40,
        "transit": 0.20,
        "international_demand": 0.15,
        "affordability": 0.15
    },

    "low_risk": {
        "yield": 0.15,
        "growth": 0.15,
        "transit": 0.25,
        "international_demand": 0.25,
        "affordability": 0.20
    }
}


# ============================================================
# 2. BASIC HELPERS
# ============================================================

def safe_number(value, default=None):
    try:
        number = float(value)

        if math.isnan(number) or math.isinf(number):
            return default

        return number

    except (TypeError, ValueError):
        return default


def resolve_investment_strategy(
    parsed=None,
    selected_strategy="balanced"
):
    """
    Prefer an investment strategy detected directly from the user's
    question. Otherwise use the structured UI selection.
    """

    valid_strategies = {
        "balanced",
        "income",
        "growth",
        "low_risk"
    }

    if parsed:

        detected = parsed.get(
            "investment_strategies",
            []
        )

        # Balanced is intentionally weaker than a clearly detected
        # income, growth or low-risk objective.
        for strategy in [
            "income",
            "growth",
            "low_risk",
            "balanced"
        ]:

            if strategy in detected:
                return strategy

    selected_strategy = normalize_text(
        selected_strategy
    ).replace("-", "_").replace(" ", "_")

    if selected_strategy in valid_strategies:
        return selected_strategy

    return "balanced"


# ============================================================
# 3. YIELD NORMALIZATION
# ============================================================

def get_yield_bounds():

    yields = [
        float(data["gross_yield"])
        for data in BANGKOK_ZONES.values()
    ]

    return min(yields), max(yields)


YIELD_MIN, YIELD_MAX = get_yield_bounds()


def calculate_yield_score(gross_yield):
    """
    Normalize database gross yield to a 0-10 screening score.
    """

    gross_yield = safe_number(
        gross_yield,
        default=YIELD_MIN
    )

    if YIELD_MAX == YIELD_MIN:
        return 5.0

    score = (
        (gross_yield - YIELD_MIN)
        / (YIELD_MAX - YIELD_MIN)
        * 10
    )

    return round(
        max(0, min(10, score)),
        2
    )


# ============================================================
# 4. BUDGET / AFFORDABILITY LOGIC
# ============================================================

def calculate_modeled_unit_cost(
    midpoint_price_sqm,
    target_size_sqm
):

    price = safe_number(
        midpoint_price_sqm
    )

    size = safe_number(
        target_size_sqm
    )

    if price is None or size is None:
        return None

    if price <= 0 or size <= 0:
        return None

    return price * size


def classify_budget_fit(
    modeled_cost,
    budget_million_thb
):

    budget_million = safe_number(
        budget_million_thb
    )

    if (
        modeled_cost is None
        or budget_million is None
        or budget_million <= 0
    ):
        return "Budget not available"

    budget_thb = (
        budget_million
        * 1_000_000
    )

    ratio = (
        modeled_cost
        / budget_thb
    )

    if ratio <= 1.00:
        return "Within budget"

    elif ratio <= 1.10:
        return "Slightly above budget"

    elif ratio <= 1.25:
        return "Above budget"

    else:
        return "Materially above budget"


def calculate_affordability_score(
    modeled_cost,
    budget_million_thb
):
    """
    0-10 affordability score.

    Within-budget properties receive a positive score reflecting
    remaining budget headroom.

    Over-budget properties are penalized progressively.
    """

    budget_million = safe_number(
        budget_million_thb
    )

    if (
        modeled_cost is None
        or budget_million is None
        or budget_million <= 0
    ):
        return 5.0

    budget_thb = (
        budget_million
        * 1_000_000
    )

    ratio = (
        modeled_cost
        / budget_thb
    )

    if ratio <= 1:

        # Exact budget = 6/10.
        # More headroom improves the score progressively.
        score = (
            6
            + 4 * (1 - ratio)
        )

    else:

        # Penalize budget overruns relatively quickly.
        score = (
            6
            - 12 * (ratio - 1)
        )

    return round(
        max(0, min(10, score)),
        2
    )


# ============================================================
# 5. SCORE ONE BANGKOK ZONE
# ============================================================

def score_bangkok_zone(
    zone_name,
    budget_million_thb=8,
    target_size_sqm=40,
    strategy="balanced"
):

    zone = BANGKOK_ZONES.get(
        zone_name
    )

    if zone is None:
        return None

    strategy = resolve_investment_strategy(
        parsed=None,
        selected_strategy=strategy
    )

    weights = STRATEGY_WEIGHTS[
        strategy
    ]

    modeled_cost = calculate_modeled_unit_cost(
        zone["midpoint_price_sqm"],
        target_size_sqm
    )

    yield_score = calculate_yield_score(
        zone["gross_yield"]
    )

    affordability_score = (
        calculate_affordability_score(
            modeled_cost,
            budget_million_thb
        )
    )

    component_scores = {

        "yield": yield_score,

        "growth": float(
            zone["growth_score"]
        ),

        "transit": float(
            zone["transit_score"]
        ),

        "international_demand": float(
            zone["international_demand_score"]
        ),

        "affordability": affordability_score
    }

    weighted_score_10 = sum(
        component_scores[key]
        * weights[key]
        for key in weights
    )

    final_score_100 = round(
        weighted_score_10 * 10,
        1
    )

    budget_status = classify_budget_fit(
        modeled_cost,
        budget_million_thb
    )

    budget_thb = (
        safe_number(
            budget_million_thb
        )
        * 1_000_000
        if safe_number(
            budget_million_thb
        ) is not None
        else None
    )

    unused_budget = None

    if (
        budget_thb is not None
        and modeled_cost is not None
    ):
        unused_budget = (
            budget_thb
            - modeled_cost
        )

    return {

        "zone": zone_name,

        "strategy": strategy,

        "investment_score": final_score_100,

        "midpoint_price_sqm": zone[
            "midpoint_price_sqm"
        ],

        "gross_yield": zone[
            "gross_yield"
        ],

        "modeled_unit_cost": modeled_cost,

        "budget_status": budget_status,

        "unused_budget": unused_budget,

        "component_scores": component_scores,

        "weights": weights,

        "transit_score": zone[
            "transit_score"
        ],

        "growth_score": zone[
            "growth_score"
        ],

        "international_demand_score": zone[
            "international_demand_score"
        ],

        "transit": zone[
            "transit"
        ],

        "property_focus": zone[
            "property_focus"
        ],

        "neighborhood": zone[
            "neighborhood"
        ],

        "investor_profile": zone[
            "investor_profile"
        ],

        "flood": zone[
            "flood"
        ],

        "soil": zone[
            "soil"
        ]
    }


# ============================================================
# 6. RANK ALL BANGKOK ZONES
# ============================================================

def rank_bangkok_zones(
    budget_million_thb=8,
    target_size_sqm=40,
    strategy="balanced"
):

    results = []

    for zone_name in BANGKOK_ZONES:

        result = score_bangkok_zone(
            zone_name=zone_name,
            budget_million_thb=budget_million_thb,
            target_size_sqm=target_size_sqm,
            strategy=strategy
        )

        if result is not None:
            results.append(result)

    results = sorted(
        results,
        key=lambda x: x[
            "investment_score"
        ],
        reverse=True
    )

    for rank, result in enumerate(
        results,
        start=1
    ):
        result["rank"] = rank

    return results


# ============================================================
# 7. GET INDIVIDUAL ZONE RESULT + RANK
# ============================================================

def get_zone_screening_result(
    zone_name,
    budget_million_thb=8,
    target_size_sqm=40,
    strategy="balanced"
):

    ranking = rank_bangkok_zones(
        budget_million_thb=budget_million_thb,
        target_size_sqm=target_size_sqm,
        strategy=strategy
    )

    for result in ranking:

        if result["zone"] == zone_name:
            return result

    return None


# ============================================================
# 8. FORMAT MONEY
# ============================================================

def format_thb(value):

    value = safe_number(value)

    if value is None:
        return "Not available"

    return (
        f"THB {value:,.0f}"
    )


# ============================================================
# 9. SCORING METHODOLOGY DESCRIPTION
# ============================================================

def describe_scoring_method(strategy):

    strategy = resolve_investment_strategy(
        parsed=None,
        selected_strategy=strategy
    )

    weights = STRATEGY_WEIGHTS[
        strategy
    ]

    return (
        f"The {strategy.replace('_', ' ')} strategy combines "
        f"gross-yield screening ({weights['yield']:.0%}), "
        f"growth ({weights['growth']:.0%}), "
        f"transit accessibility ({weights['transit']:.0%}), "
        f"international demand ({weights['international_demand']:.0%}) "
        f"and affordability / budget fit ({weights['affordability']:.0%})."
    )


# ============================================================
# 10. DATABASE / MODEL VALIDATION
# ============================================================

scoring_validation_errors = []

for strategy_name, weights in STRATEGY_WEIGHTS.items():

    total_weight = sum(
        weights.values()
    )

    if abs(
        total_weight - 1.0
    ) > 0.0001:

        scoring_validation_errors.append(
            f"{strategy_name}: weights sum to {total_weight}"
        )


test_ranking = rank_bangkok_zones(
    budget_million_thb=8,
    target_size_sqm=40,
    strategy="balanced"
)

if len(test_ranking) != len(
    BANGKOK_ZONES
):

    scoring_validation_errors.append(
        "Ranking does not contain all Bangkok zones."
    )


if scoring_validation_errors:

    print(
        "⚠ Investment scoring validation problems:"
    )

    for error in scoring_validation_errors:
        print("-", error)

else:

    print(
        "✓ Bangkok investment scoring engine loaded successfully"
    )

    print(
        f"✓ {len(STRATEGY_WEIGHTS)} investment strategies configured"
    )

    print(
        f"✓ {len(test_ranking)} Bangkok zones successfully scored"
    )

    print(
        "✓ Strategy weights validated"
    )

    print(
        "✓ Investment scoring engine validation passed"
    )

✓ Bangkok investment scoring engine loaded successfully
✓ 4 investment strategies configured
✓ 20 Bangkok zones successfully scored
✓ Strategy weights validated
✓ Investment scoring engine validation passed


In [16]:
# CELL 8A - Scoring Engine Stress Test

scoring_test_cases = [
    {
        "name": "Balanced investor",
        "budget": 8,
        "size": 40,
        "strategy": "balanced"
    },
    {
        "name": "Income investor",
        "budget": 8,
        "size": 40,
        "strategy": "income"
    },
    {
        "name": "Growth investor",
        "budget": 8,
        "size": 40,
        "strategy": "growth"
    },
    {
        "name": "Low-risk investor",
        "budget": 8,
        "size": 40,
        "strategy": "low_risk"
    },
    {
        "name": "Tight-budget income investor",
        "budget": 5,
        "size": 40,
        "strategy": "income"
    }
]


for i, case in enumerate(scoring_test_cases, 1):

    print("\n" + "=" * 90)
    print(f"TEST {i}: {case['name']}")
    print("=" * 90)

    print(
        f"Budget: THB {case['budget']} million | "
        f"Size: {case['size']} sqm | "
        f"Strategy: {case['strategy']}"
    )

    print()
    print(describe_scoring_method(case["strategy"]))
    print()

    ranking = rank_bangkok_zones(
        budget_million_thb=case["budget"],
        target_size_sqm=case["size"],
        strategy=case["strategy"]
    )

    print("TOP 5")
    print("-" * 90)

    for result in ranking[:5]:

        print(
            f"{result['rank']}. "
            f"{result['zone']} | "
            f"Score: {result['investment_score']}/100 | "
            f"Yield: {result['gross_yield']:.2f}% | "
            f"Growth: {result['growth_score']}/10 | "
            f"Transit: {result['transit_score']}/10 | "
            f"Intl Demand: {result['international_demand_score']}/10 | "
            f"Cost: {format_thb(result['modeled_unit_cost'])} | "
            f"{result['budget_status']}"
        )

    print()
    print("BOTTOM 3")
    print("-" * 90)

    for result in ranking[-3:]:

        print(
            f"{result['rank']}. "
            f"{result['zone']} | "
            f"Score: {result['investment_score']}/100 | "
            f"Cost: {format_thb(result['modeled_unit_cost'])} | "
            f"{result['budget_status']}"
        )


TEST 1: Balanced investor
Budget: THB 8 million | Size: 40 sqm | Strategy: balanced

The balanced strategy combines gross-yield screening (20%), growth (25%), transit accessibility (20%), international demand (15%) and affordability / budget fit (20%).

TOP 5
------------------------------------------------------------------------------------------
1. Bang Sue - Tao Poon | Score: 84.1/100 | Yield: 6.25% | Growth: 8/10 | Transit: 10/10 | Intl Demand: 6/10 | Cost: THB 4,100,000 | Within budget
2. On Nut - Phra Khanong | Score: 83.1/100 | Yield: 6.05% | Growth: 8/10 | Transit: 9/10 | Intl Demand: 8/10 | Cost: THB 4,600,000 | Within budget
3. Punnawithi - Udom Suk | Score: 82.7/100 | Yield: 6.05% | Growth: 9/10 | Transit: 8/10 | Intl Demand: 7/10 | Cost: THB 4,000,000 | Within budget
4. Rama 9 - Ratchada | Score: 80.4/100 | Yield: 5.50% | Growth: 9/10 | Transit: 9/10 | Intl Demand: 8/10 | Cost: THB 5,600,000 | Within budget
5. Huai Khwang | Score: 79.2/100 | Yield: 5.75% | Growth: 8/10 | 

In [17]:
# CELL 9 - Bangkok Area Recommendation Response Engine

def build_area_recommendation_response(parsed, budget_million_thb=8, target_size_sqm=40, strategy=None):
    """
    Builds a structured Bangkok investment-area recommendation.
    Uses the validated scoring engine from CELL 8.
    """

    # ---------------------------------------------------------
    # 1. SAFE INPUT HANDLING
    # ---------------------------------------------------------

    try:
        budget = float(budget_million_thb)
        if budget <= 0:
            budget = 8.0
    except (TypeError, ValueError):
        budget = 8.0

    try:
        size = float(target_size_sqm)
        if size <= 0:
            size = 40.0
    except (TypeError, ValueError):
        size = 40.0

    valid_strategies = {"balanced", "income", "growth", "low_risk"}

    if strategy not in valid_strategies:
        detected_strategies = parsed.get("investment_strategies", [])

        if detected_strategies:
            candidate = detected_strategies[0]
            strategy = candidate if candidate in valid_strategies else "balanced"
        else:
            strategy = "balanced"

    # ---------------------------------------------------------
    # 2. RANK BANGKOK ZONES
    # ---------------------------------------------------------

    ranking = rank_bangkok_zones(
        budget_million_thb=budget,
        target_size_sqm=size,
        strategy=strategy
    )

    # Prefer zones that fit the stated budget
    within_budget = [
        result for result in ranking
        if result["budget_status"] == "Within budget"
    ]

    if len(within_budget) >= 5:
        recommendations = within_budget[:5]
    else:
        recommendations = ranking[:5]

    # ---------------------------------------------------------
    # 3. RESPONSE HEADER
    # ---------------------------------------------------------

    response = []

    response.append("# TH Bangkok Foreign Investor AI")
    response.append("")
    response.append("## Bangkok Investment Area Screening")
    response.append("")

    response.append(
        f"**Investor profile used:** "
        f"THB {budget:g} million budget | "
        f"{size:g} sqm target size | "
        f"{strategy.replace('_', ' ').title()} strategy"
    )

    response.append("")

    # ---------------------------------------------------------
    # 4. STRATEGY INTERPRETATION
    # ---------------------------------------------------------

    strategy_descriptions = {
        "balanced":
            "The screening balances rental yield, growth potential, "
            "transit accessibility, international demand and affordability.",

        "income":
            "The screening places greater emphasis on gross rental yield "
            "and affordability while retaining market-demand and transit factors.",

        "growth":
            "The screening places greater emphasis on expected area-level "
            "growth potential and transit accessibility.",

        "low_risk":
            "The screening places greater emphasis on transit accessibility, "
            "international demand and affordability as defensive investment indicators."
    }

    response.append("### Investment Strategy")
    response.append(strategy_descriptions[strategy])
    response.append("")

    # ---------------------------------------------------------
    # 5. TOP RECOMMENDATIONS
    # ---------------------------------------------------------

    response.append("## Top Bangkok Areas")
    response.append("")

    for result in recommendations:

        response.append(
            f"### {result['rank']}. {result['zone']}"
        )

        response.append(
            f"**Investment score:** {result['investment_score']}/100"
        )

        response.append(
            f"**Indicative gross rental yield:** "
            f"{result['gross_yield']:.2f}%"
        )

        response.append(
            f"**Growth score:** {result['growth_score']}/10"
        )

        response.append(
            f"**Transit accessibility:** {result['transit_score']}/10"
        )

        response.append(
            f"**International demand:** "
            f"{result['international_demand_score']}/10"
        )

        response.append(
            f"**Modeled cost for {size:g} sqm:** "
            f"{format_thb(result['modeled_unit_cost'])}"
        )

        response.append(
            f"**Budget assessment:** {result['budget_status']}"
        )

        response.append("")

    # ---------------------------------------------------------
    # 6. INVESTMENT INTERPRETATION
    # ---------------------------------------------------------

    top = recommendations[0]

    response.append("## Investment Interpretation")
    response.append("")

    response.append(
        f"Under the selected **{strategy.replace('_', ' ')}** strategy, "
        f"**{top['zone']}** receives the highest screening score among "
        f"the evaluated Bangkok zones that best fit the stated profile."
    )

    response.append("")

    response.append(
        "The ranking should be interpreted as an **area-level screening tool**, "
        "not as a recommendation to purchase a specific property. "
        "Actual investment performance depends on the individual building, "
        "unit quality, purchase price, achievable rent, vacancy, operating costs, "
        "legal status and transaction terms."
    )

    # ---------------------------------------------------------
    # 7. BUDGET NOTE
    # ---------------------------------------------------------

    response.append("")
    response.append("## Budget Fit")
    response.append("")

    affordable_count = len(within_budget)

    response.append(
        f"{affordable_count} of the {len(ranking)} screened Bangkok zones "
        f"have a modeled {size:g} sqm unit cost within the stated "
        f"THB {budget:g} million budget."
    )

    response.append("")

    response.append(
        "Modeled unit cost is calculated from area-level indicative price-per-sqm "
        "data and the selected target size. It is not a property valuation or "
        "a guarantee that a suitable unit is available at that price."
    )

    # ---------------------------------------------------------
    # 8. DUE DILIGENCE
    # ---------------------------------------------------------

    response.append("")
    response.append("## Before Investing")
    response.append("")

    response.append(
        "- Verify the actual asking price against recent comparable properties."
    )
    response.append(
        "- Verify achievable rent, vacancy assumptions and recurring ownership costs."
    )
    response.append(
        "- Confirm foreign ownership eligibility and condominium foreign quota where applicable."
    )
    response.append(
        "- Review title, legal records, building condition and condominium juristic-person information."
    )
    response.append(
        "- Conduct property-specific flood, drainage, soil and engineering due diligence where material."
    )
    response.append(
        "- Consider transaction costs, taxes, financing and exit liquidity before making an investment decision."
    )

    # ---------------------------------------------------------
    # 9. MODEL LIMITATION
    # ---------------------------------------------------------

    response.append("")
    response.append("## Model Limitation")
    response.append("")

    response.append(
        "Area prices, yields, growth, transit and demand indicators are "
        "screening-level inputs. They are not transaction-level appraisal evidence. "
        "A high area score does not override property-specific legal, technical "
        "or financial concerns."
    )

    # ---------------------------------------------------------
    # 10. DISCLAIMER
    # ---------------------------------------------------------

    response.append("")
    response.append("## Disclaimer")
    response.append("")
    response.append(LEGAL_DISCLAIMER)

    return "\n".join(response)


print("✓ Bangkok area recommendation response engine loaded successfully")

✓ Bangkok area recommendation response engine loaded successfully


In [18]:
# CELL 9A - Area Recommendation Response Test

test_question = (
    "I have THB 5 million and want good rental income. "
    "Which Bangkok areas should I consider?"
)

parsed = parse_investment_query(test_question)

route = route_investment_query(parsed)

print("Detected route:", route["primary_route"])
print()

answer = build_area_recommendation_response(
    parsed=parsed,
    budget_million_thb=5,
    target_size_sqm=40,
    strategy="income"
)

display(Markdown(answer))

Detected route: area_recommendation



# TH Bangkok Foreign Investor AI

## Bangkok Investment Area Screening

**Investor profile used:** THB 5 million budget | 40 sqm target size | Income strategy

### Investment Strategy
The screening places greater emphasis on gross rental yield and affordability while retaining market-demand and transit factors.

## Top Bangkok Areas

### 1. Bang Sue - Tao Poon
**Investment score:** 82.5/100
**Indicative gross rental yield:** 6.25%
**Growth score:** 8/10
**Transit accessibility:** 10/10
**International demand:** 6/10
**Modeled cost for 40 sqm:** THB 4,100,000
**Budget assessment:** Within budget

### 2. On Nut - Phra Khanong
**Investment score:** 80.3/100
**Indicative gross rental yield:** 6.05%
**Growth score:** 8/10
**Transit accessibility:** 9/10
**International demand:** 8/10
**Modeled cost for 40 sqm:** THB 4,600,000
**Budget assessment:** Within budget

### 3. Talat Phlu - Wutthakat
**Investment score:** 80.3/100
**Indicative gross rental yield:** 6.35%
**Growth score:** 7/10
**Transit accessibility:** 8/10
**International demand:** 5/10
**Modeled cost for 40 sqm:** THB 3,100,000
**Budget assessment:** Within budget

### 4. Punnawithi - Udom Suk
**Investment score:** 79.5/100
**Indicative gross rental yield:** 6.05%
**Growth score:** 9/10
**Transit accessibility:** 8/10
**International demand:** 7/10
**Modeled cost for 40 sqm:** THB 4,000,000
**Budget assessment:** Within budget

### 5. Ramkhamhaeng - Bang Kapi
**Investment score:** 76.7/100
**Indicative gross rental yield:** 6.15%
**Growth score:** 8/10
**Transit accessibility:** 7/10
**International demand:** 5/10
**Modeled cost for 40 sqm:** THB 3,300,000
**Budget assessment:** Within budget

## Investment Interpretation

Under the selected **income** strategy, **Bang Sue - Tao Poon** receives the highest screening score among the evaluated Bangkok zones that best fit the stated profile.

The ranking should be interpreted as an **area-level screening tool**, not as a recommendation to purchase a specific property. Actual investment performance depends on the individual building, unit quality, purchase price, achievable rent, vacancy, operating costs, legal status and transaction terms.

## Budget Fit

11 of the 20 screened Bangkok zones have a modeled 40 sqm unit cost within the stated THB 5 million budget.

Modeled unit cost is calculated from area-level indicative price-per-sqm data and the selected target size. It is not a property valuation or a guarantee that a suitable unit is available at that price.

## Before Investing

- Verify the actual asking price against recent comparable properties.
- Verify achievable rent, vacancy assumptions and recurring ownership costs.
- Confirm foreign ownership eligibility and condominium foreign quota where applicable.
- Review title, legal records, building condition and condominium juristic-person information.
- Conduct property-specific flood, drainage, soil and engineering due diligence where material.
- Consider transaction costs, taxes, financing and exit liquidity before making an investment decision.

## Model Limitation

Area prices, yields, growth, transit and demand indicators are screening-level inputs. They are not transaction-level appraisal evidence. A high area score does not override property-specific legal, technical or financial concerns.

## Disclaimer

This chatbot is an educational investment-screening prototype and does not provide legal, tax, engineering, valuation or financial advice. Regulations, market conditions and transaction requirements may change. Users should verify current requirements and property-specific information with competent authorities, qualified legal advisers, banks, engineers, valuers and other professional advisers.

In [19]:
# CELL 10 - Single-Area Screening & Area Comparison Response Engine


def _resolve_profile_inputs(
    parsed,
    budget_million_thb=8,
    target_size_sqm=40,
    strategy="balanced"
):
    """
    Use values detected in the user's question where available.
    Otherwise fall back to structured UI/default values.
    """

    detected_budget = parsed.get("budget_million_thb")
    detected_size = parsed.get("property_size_sqm")

    budget = (
        detected_budget
        if detected_budget is not None
        else budget_million_thb
    )

    size = (
        detected_size
        if detected_size is not None
        else target_size_sqm
    )

    final_strategy = resolve_investment_strategy(
        parsed=parsed,
        selected_strategy=strategy
    )

    return (
        float(budget),
        float(size),
        final_strategy
    )


def _format_zone_screening_block(result, size):
    """
    Format one Bangkok zone as a structured screening block.
    """

    if result is None:
        return (
            "No structured screening result is available "
            "for this Bangkok area."
        )

    lines = []

    lines.append(
        f"### {result['zone']}"
    )
    lines.append("")

    lines.append(
        f"**Investment score:** "
        f"{result['investment_score']}/100"
    )

    lines.append(
        f"**Overall ranking:** "
        f"{result['rank']} of {len(BANGKOK_ZONES)} covered areas"
    )

    lines.append("")

    lines.append("#### Financial Fit")

    lines.append(
        f"- Indicative midpoint price: "
        f"{format_thb(result['midpoint_price_sqm'])} per sqm"
    )

    lines.append(
        f"- Modeled {size:g} sqm unit cost: "
        f"{format_thb(result['modeled_unit_cost'])}"
    )

    lines.append(
        f"- Budget assessment: "
        f"**{result['budget_status']}**"
    )

    lines.append(
        f"- Indicative gross rental yield: "
        f"{result['gross_yield']:.2f}%"
    )

    lines.append("")

    lines.append("#### Market Characteristics")

    lines.append(
        f"- Growth score: "
        f"{result['growth_score']}/10"
    )

    lines.append(
        f"- Transit score: "
        f"{result['transit_score']}/10"
    )

    lines.append(
        f"- International demand score: "
        f"{result['international_demand_score']}/10"
    )

    lines.append(
        f"- Transit context: "
        f"{result['transit']}"
    )

    lines.append(
        f"- Typical property focus: "
        f"{result['property_focus']}"
    )

    lines.append(
        f"- Neighborhood profile: "
        f"{result['neighborhood']}"
    )

    lines.append(
        f"- Typical investor profile: "
        f"{result['investor_profile']}"
    )

    lines.append("")

    lines.append("#### Technical Screening")

    lines.append(
        f"- Flood / drainage: "
        f"{result['flood']}"
    )

    lines.append(
        f"- Soil / engineering: "
        f"{result['soil']}"
    )

    return "\n".join(lines)


# ============================================================
# 1. SINGLE-AREA SCREENING
# ============================================================

def build_single_area_screening_response(
    parsed,
    budget_million_thb=8,
    target_size_sqm=40,
    strategy="balanced"
):
    """
    Analyse one explicitly mentioned Bangkok area.
    """

    locations = parsed.get(
        "locations",
        []
    )

    response = []

    response.append(
        "# TH Bangkok Foreign Investor AI"
    )
    response.append("")

    response.append(
        "## Bangkok Area Investment Screening"
    )
    response.append("")

    if not locations:

        response.append(
            "No covered Bangkok area could be identified "
            "from the question."
        )

        response.append("")

        response.append(
            "Please specify a Bangkok area such as "
            "**On Nut, Bang Kapi, Rama 9, Ari or Sathorn**."
        )

        response.append("")

        response.append(
            "## Disclaimer"
        )
        response.append("")
        response.append(
            LEGAL_DISCLAIMER
        )

        return "\n".join(response)

    zone_name = locations[0]

    budget, size, final_strategy = (
        _resolve_profile_inputs(
            parsed,
            budget_million_thb,
            target_size_sqm,
            strategy
        )
    )

    result = get_zone_screening_result(
        zone_name=zone_name,
        budget_million_thb=budget,
        target_size_sqm=size,
        strategy=final_strategy
    )

    response.append(
        f"**Investor profile used:** "
        f"THB {budget:g} million budget | "
        f"{size:g} sqm target size | "
        f"{final_strategy.replace('_', ' ').title()} strategy"
    )

    response.append("")

    response.append(
        _format_zone_screening_block(
            result,
            size
        )
    )

    response.append("")
    response.append(
        "## Investment Interpretation"
    )
    response.append("")

    if result is not None:

        response.append(
            f"**{zone_name}** receives an area-level "
            f"screening score of "
            f"**{result['investment_score']}/100** under "
            f"the selected {final_strategy.replace('_', ' ')} "
            f"strategy."
        )

        response.append("")

        if result["budget_status"] == "Within budget":

            response.append(
                "The modeled unit cost fits within the stated budget."
            )

        else:

            response.append(
                f"The modeled unit cost is classified as "
                f"**{result['budget_status']}**, so budget fit "
                "requires additional attention."
            )

        response.append("")

        response.append(
            "The result should be interpreted as an "
            "**area-screening indication**, not as proof that "
            "a particular unit is a good investment."
        )

    response.append("")
    response.append(
        "## Before Investing"
    )
    response.append("")

    response.append(
        format_bullet_list([
            "Compare the actual purchase price with relevant property-level comparables.",
            "Verify achievable rent, vacancy and recurring ownership costs.",
            "Confirm foreign-investor legal eligibility and title status.",
            "Review building quality, common-area management and planned repairs.",
            "Check street- and property-level flood exposure.",
            "Review structural, foundation and engineering information where relevant.",
            "Evaluate resale liquidity and likely tenant or buyer demand."
        ])
    )

    response.append("")
    response.append(
        "## Model Limitation"
    )
    response.append("")

    response.append(
        DATA_LIMITATION_NOTE
    )

    response.append("")
    response.append(
        "## Disclaimer"
    )
    response.append("")

    response.append(
        LEGAL_DISCLAIMER
    )

    return "\n".join(response)


# ============================================================
# 2. MULTI-AREA COMPARISON
# ============================================================

def build_area_comparison_response(
    parsed,
    budget_million_thb=8,
    target_size_sqm=40,
    strategy="balanced"
):
    """
    Directly compare two or more detected Bangkok areas.
    """

    locations = parsed.get(
        "locations",
        []
    )

    response = []

    response.append(
        "# TH Bangkok Foreign Investor AI"
    )
    response.append("")

    response.append(
        "## Bangkok Area Comparison"
    )
    response.append("")

    if len(locations) < 2:

        response.append(
            "A direct comparison requires at least two "
            "recognized Bangkok areas."
        )

        response.append("")

        response.append(
            "Please name two or more covered areas."
        )

        response.append("")

        response.append(
            "## Disclaimer"
        )
        response.append("")
        response.append(
            LEGAL_DISCLAIMER
        )

        return "\n".join(response)

    budget, size, final_strategy = (
        _resolve_profile_inputs(
            parsed,
            budget_million_thb,
            target_size_sqm,
            strategy
        )
    )

    results = []

    for zone_name in locations:

        result = get_zone_screening_result(
            zone_name=zone_name,
            budget_million_thb=budget,
            target_size_sqm=size,
            strategy=final_strategy
        )

        if result is not None:
            results.append(result)

    response.append(
        f"**Investor profile used:** "
        f"THB {budget:g} million budget | "
        f"{size:g} sqm target size | "
        f"{final_strategy.replace('_', ' ').title()} strategy"
    )

    response.append("")

    response.append(
        f"**Areas detected:** "
        + " vs. ".join(
            result["zone"]
            for result in results
        )
    )

    response.append("")

    response.append(
        "## Comparison Summary"
    )
    response.append("")

    response.append(
        "| Area | Score | Yield | Growth | Transit | Intl. Demand | "
        f"Modeled {size:g} sqm Cost | Budget |"
    )

    response.append(
        "|---|---:|---:|---:|---:|---:|---:|---|"
    )

    for result in results:

        response.append(
            f"| {result['zone']} "
            f"| {result['investment_score']}/100 "
            f"| {result['gross_yield']:.2f}% "
            f"| {result['growth_score']}/10 "
            f"| {result['transit_score']}/10 "
            f"| {result['international_demand_score']}/10 "
            f"| {format_thb(result['modeled_unit_cost'])} "
            f"| {result['budget_status']} |"
        )

    response.append("")
    response.append(
        "## Detailed Area Profiles"
    )
    response.append("")

    for result in results:

        response.append(
            _format_zone_screening_block(
                result,
                size
            )
        )

        response.append("")

    response.append(
        "## Comparison Interpretation"
    )
    response.append("")

    ranked_results = sorted(
        results,
        key=lambda x: x[
            "investment_score"
        ],
        reverse=True
    )

    if len(ranked_results) >= 2:

        leader = ranked_results[0]
        second = ranked_results[1]

        difference = round(
            leader["investment_score"]
            - second["investment_score"],
            1
        )

        response.append(
            f"Under the selected "
            f"**{final_strategy.replace('_', ' ')}** strategy, "
            f"**{leader['zone']}** receives the highest score "
            f"among the compared areas at "
            f"**{leader['investment_score']}/100**."
        )

        response.append("")

        response.append(
            f"The score difference versus "
            f"**{second['zone']}** is "
            f"**{difference} points**."
        )

        response.append("")

        response.append(
            "That result does not mean the higher-ranked area "
            "will automatically produce the better individual "
            "investment. A lower-ranked area may still offer a "
            "superior property if the actual purchase price, unit "
            "quality, rent, building condition or transaction terms "
            "are more attractive."
        )

    response.append("")
    response.append(
        "## Decision Factors Beyond the Score"
    )
    response.append("")

    response.append(
        format_bullet_list([
            "Actual purchase price and comparable transactions",
            "Achievable rent and vacancy",
            "Property and building quality",
            "Foreign ownership eligibility and legal status",
            "Flood, drainage, soil and engineering risk",
            "Tenant profile and resale liquidity",
            "Transaction costs and holding period"
        ])
    )

    response.append("")
    response.append(
        "## Model Limitation"
    )
    response.append("")

    response.append(
        DATA_LIMITATION_NOTE
    )

    response.append("")
    response.append(
        "## Disclaimer"
    )
    response.append("")

    response.append(
        LEGAL_DISCLAIMER
    )

    return "\n".join(response)


print(
    "✓ Single-area screening & comparison response engine loaded successfully"
)

✓ Single-area screening & comparison response engine loaded successfully


In [20]:
# CELL 10A - Single-Area & Comparison Stress Test

test_questions = [
    "Is Bang Kapi attractive for rental income?",
    "Compare Bang Kapi with On Nut for rental income."
]

for i, question in enumerate(test_questions, 1):

    print("\n" + "=" * 90)
    print(f"TEST {i}: {question}")
    print("=" * 90)

    parsed = parse_investment_query(question)
    route = route_investment_query(parsed)

    print(f"Detected route: {route['primary_route']}")
    print(f"Locations: {parsed.get('locations', [])}")
    print(f"Strategies: {parsed.get('investment_strategies', [])}")
    print()

    if route["primary_route"] == "property_screening":
        answer = build_single_area_screening_response(
            parsed=parsed,
            budget_million_thb=8,
            target_size_sqm=40,
            strategy="balanced"
        )

    elif route["primary_route"] == "area_comparison":
        answer = build_area_comparison_response(
            parsed=parsed,
            budget_million_thb=8,
            target_size_sqm=40,
            strategy="balanced"
        )

    else:
        answer = (
            f"Unexpected route: {route['primary_route']}"
        )

    display(Markdown(answer))


TEST 1: Is Bang Kapi attractive for rental income?
Detected route: property_screening
Locations: ['Ramkhamhaeng - Bang Kapi']
Strategies: ['income']



# TH Bangkok Foreign Investor AI

## Bangkok Area Investment Screening

**Investor profile used:** THB 8 million budget | 40 sqm target size | Income strategy

### Ramkhamhaeng - Bang Kapi

**Investment score:** 79.2/100
**Overall ranking:** 6 of 20 covered areas

#### Financial Fit
- Indicative midpoint price: THB 82,500 per sqm
- Modeled 40 sqm unit cost: THB 3,300,000
- Budget assessment: **Within budget**
- Indicative gross rental yield: 6.15%

#### Market Characteristics
- Growth score: 8/10
- Transit score: 7/10
- International demand score: 5/10
- Transit context: Airport Rail Link / rail expansion / major road connections
- Typical property focus: Condominium / Residential
- Neighborhood profile: Universities, large residential population and local retail demand
- Typical investor profile: Budget / local rental demand

#### Technical Screening
- Flood / drainage: Flood and drainage due diligence is particularly important at street level
- Soil / engineering: Soft clay; low-lying site conditions require engineering review

## Investment Interpretation

**Ramkhamhaeng - Bang Kapi** receives an area-level screening score of **79.2/100** under the selected income strategy.

The modeled unit cost fits within the stated budget.

The result should be interpreted as an **area-screening indication**, not as proof that a particular unit is a good investment.

## Before Investing

- Compare the actual purchase price with relevant property-level comparables.
- Verify achievable rent, vacancy and recurring ownership costs.
- Confirm foreign-investor legal eligibility and title status.
- Review building quality, common-area management and planned repairs.
- Check street- and property-level flood exposure.
- Review structural, foundation and engineering information where relevant.
- Evaluate resale liquidity and likely tenant or buyer demand.

## Model Limitation

Area prices, yields, flood, soil and market indicators are screening-level inputs. They are not transaction-level appraisal evidence and should not replace property-specific legal, technical and financial due diligence.

## Disclaimer

This chatbot is an educational investment-screening prototype and does not provide legal, tax, engineering, valuation or financial advice. Regulations, market conditions and transaction requirements may change. Users should verify current requirements and property-specific information with competent authorities, qualified legal advisers, banks, engineers, valuers and other professional advisers.


TEST 2: Compare Bang Kapi with On Nut for rental income.
Detected route: area_comparison
Locations: ['Ramkhamhaeng - Bang Kapi', 'On Nut - Phra Khanong']
Strategies: ['income']



# TH Bangkok Foreign Investor AI

## Bangkok Area Comparison

**Investor profile used:** THB 8 million budget | 40 sqm target size | Income strategy

**Areas detected:** Ramkhamhaeng - Bang Kapi vs. On Nut - Phra Khanong

## Comparison Summary

| Area | Score | Yield | Growth | Transit | Intl. Demand | Modeled 40 sqm Cost | Budget |
|---|---:|---:|---:|---:|---:|---:|---|
| Ramkhamhaeng - Bang Kapi | 79.2/100 | 6.15% | 8/10 | 7/10 | 5/10 | THB 3,300,000 | Within budget |
| On Nut - Phra Khanong | 83.7/100 | 6.05% | 8/10 | 9/10 | 8/10 | THB 4,600,000 | Within budget |

## Detailed Area Profiles

### Ramkhamhaeng - Bang Kapi

**Investment score:** 79.2/100
**Overall ranking:** 6 of 20 covered areas

#### Financial Fit
- Indicative midpoint price: THB 82,500 per sqm
- Modeled 40 sqm unit cost: THB 3,300,000
- Budget assessment: **Within budget**
- Indicative gross rental yield: 6.15%

#### Market Characteristics
- Growth score: 8/10
- Transit score: 7/10
- International demand score: 5/10
- Transit context: Airport Rail Link / rail expansion / major road connections
- Typical property focus: Condominium / Residential
- Neighborhood profile: Universities, large residential population and local retail demand
- Typical investor profile: Budget / local rental demand

#### Technical Screening
- Flood / drainage: Flood and drainage due diligence is particularly important at street level
- Soil / engineering: Soft clay; low-lying site conditions require engineering review

### On Nut - Phra Khanong

**Investment score:** 83.7/100
**Overall ranking:** 2 of 20 covered areas

#### Financial Fit
- Indicative midpoint price: THB 115,000 per sqm
- Modeled 40 sqm unit cost: THB 4,600,000
- Budget assessment: **Within budget**
- Indicative gross rental yield: 6.05%

#### Market Characteristics
- Growth score: 8/10
- Transit score: 9/10
- International demand score: 8/10
- Transit context: BTS Sukhumvit Line
- Typical property focus: Condominium
- Neighborhood profile: Value-oriented residential area with growing international tenant base
- Typical investor profile: Yield / value / first-time investor

#### Technical Screening
- Flood / drainage: Check soi-level drainage and access during heavy rainfall
- Soil / engineering: Bangkok soft-clay conditions

## Comparison Interpretation

Under the selected **income** strategy, **On Nut - Phra Khanong** receives the highest score among the compared areas at **83.7/100**.

The score difference versus **Ramkhamhaeng - Bang Kapi** is **4.5 points**.

That result does not mean the higher-ranked area will automatically produce the better individual investment. A lower-ranked area may still offer a superior property if the actual purchase price, unit quality, rent, building condition or transaction terms are more attractive.

## Decision Factors Beyond the Score

- Actual purchase price and comparable transactions
- Achievable rent and vacancy
- Property and building quality
- Foreign ownership eligibility and legal status
- Flood, drainage, soil and engineering risk
- Tenant profile and resale liquidity
- Transaction costs and holding period

## Model Limitation

Area prices, yields, flood, soil and market indicators are screening-level inputs. They are not transaction-level appraisal evidence and should not replace property-specific legal, technical and financial due diligence.

## Disclaimer

This chatbot is an educational investment-screening prototype and does not provide legal, tax, engineering, valuation or financial advice. Regulations, market conditions and transaction requirements may change. Users should verify current requirements and property-specific information with competent authorities, qualified legal advisers, banks, engineers, valuers and other professional advisers.

In [21]:
# CELL 11 - Unified Response Engine

def generate_unified_response(
    user_query,
    budget_million_thb=8,
    target_size_sqm=40,
    selected_strategy="balanced"
):
    """
    Central V4 response engine.

    Flow:
    1. Parse the user question
    2. Route the intent
    3. Call the appropriate specialized response engine
    4. Return a controlled fallback if needed
    """

    # --------------------------------------------------------
    # 1. SAFE INPUT HANDLING
    # --------------------------------------------------------

    if user_query is None:
        user_query = ""

    user_query = str(user_query).strip()

    if not user_query:
        return (
            "# TH Bangkok Foreign Investor AI\n\n"
            "## Clarification Required\n\n"
            "Please enter a real-estate investment question.\n\n"
            "Examples:\n"
            "- Can a foreigner buy land in Bangkok?\n"
            "- Which Bangkok areas are best for rental income?\n"
            "- Compare Bang Kapi with On Nut.\n"
            "- What should I check regarding flood and soil conditions?\n"
            "- What is the difference between gross rental yield and cap rate?\n"
        )

    try:
        budget = float(budget_million_thb)
        if budget <= 0:
            budget = 8.0
    except (TypeError, ValueError):
        budget = 8.0

    try:
        size = float(target_size_sqm)
        if size <= 0:
            size = 40.0
    except (TypeError, ValueError):
        size = 40.0

    # --------------------------------------------------------
    # 2. PARSE + ROUTE
    # --------------------------------------------------------

    parsed = parse_investment_query(
        user_query
    )

    route = route_investment_query(
        parsed
    )

    primary_route = route[
        "primary_route"
    ]

    # --------------------------------------------------------
    # 3. SPECIALIZED RESPONSE ROUTES
    # --------------------------------------------------------

    if primary_route in [
        "legal_restriction",
        "legal_guidance"
    ]:
        return build_legal_response(
            parsed
        )

    elif primary_route == "technical_due_diligence":
        return build_technical_response(
            parsed
        )

    elif primary_route == "valuation_definition":
        return build_valuation_definition_response(
            parsed
        )

    elif primary_route == "valuation_calculation":
        return build_valuation_calculation_response(
            parsed
        )

    elif primary_route == "area_recommendation":
        return build_area_recommendation_response(
            parsed=parsed,
            budget_million_thb=budget,
            target_size_sqm=size,
            strategy=selected_strategy
        )

    elif primary_route == "area_comparison":
        return build_area_comparison_response(
            parsed=parsed,
            budget_million_thb=budget,
            target_size_sqm=size,
            strategy=selected_strategy
        )

    elif primary_route == "property_screening":
        return build_single_area_screening_response(
            parsed=parsed,
            budget_million_thb=budget,
            target_size_sqm=size,
            strategy=selected_strategy
        )

    elif primary_route == "general_due_diligence":

        response = []

        response.append(
            "# TH Bangkok Foreign Investor AI"
        )
        response.append("")

        response.append(
            "## General Real-Estate Due Diligence"
        )
        response.append("")

        response.append(
            "A real-estate investment should normally be reviewed across "
            "legal, financial, market, technical and exit-risk dimensions."
        )
        response.append("")

        for category, items in DUE_DILIGENCE_CATEGORIES.items():

            response.append(
                f"### {category.replace('_', ' ').title()}"
            )
            response.append("")

            response.append(
                format_bullet_list(
                    items
                )
            )

            response.append("")

        response.append(
            "## Model Limitation"
        )
        response.append("")

        response.append(
            DATA_LIMITATION_NOTE
        )
        response.append("")

        response.append(
            "## Disclaimer"
        )
        response.append("")

        response.append(
            LEGAL_DISCLAIMER
        )

        return "\n".join(response)

    elif primary_route == "general_real_estate":

        response = []

        response.append(
            "# TH Bangkok Foreign Investor AI"
        )
        response.append("")

        response.append(
            "## General Real-Estate Guidance"
        )
        response.append("")

        response.append(
            "The question is within the broader real-estate domain, "
            "but it does not match one of the specialized screening "
            "modules with enough confidence."
        )

        response.append("")

        response.append(
            "For a more targeted answer, specify whether you want help with:"
        )

        response.append("")

        response.append(
            format_bullet_list([
                "Foreign ownership or legal structure",
                "Bangkok area recommendation",
                "Comparison of two or more areas",
                "Rental income or growth strategy",
                "Flood, soil or technical due diligence",
                "Valuation concepts such as NPV, IRR, NOI or Cap Rate",
                "Property-specific investment screening"
            ])
        )

        response.append("")

        response.append(
            "## Disclaimer"
        )
        response.append("")

        response.append(
            LEGAL_DISCLAIMER
        )

        return "\n".join(response)

    # --------------------------------------------------------
    # 4. CONTROLLED FALLBACK
    # --------------------------------------------------------

    response = []

    response.append(
        "# TH Bangkok Foreign Investor AI"
    )
    response.append("")

    response.append(
        "## Clarification Required"
    )
    response.append("")

    response.append(
        "I could not classify the question reliably enough to "
        "provide a specialized investment response."
    )

    response.append("")

    response.append(
        "Please clarify the property type, Bangkok area, investment "
        "objective or topic you want to analyze."
    )

    response.append("")

    response.append(
        "Examples:"
    )

    response.append("")

    response.append(
        format_bullet_list([
            "Can a foreigner buy a condominium in Bangkok?",
            "Which Bangkok areas fit a THB 5 million budget?",
            "Compare Bang Kapi with On Nut for rental income.",
            "What should I check regarding flood and soil conditions?",
            "How do I calculate NPV for a real-estate investment?"
        ])
    )

    response.append("")

    response.append(
        "## Disclaimer"
    )
    response.append("")

    response.append(
        LEGAL_DISCLAIMER
    )

    return "\n".join(response)


print("✓ Unified V4 response engine loaded successfully")

✓ Unified V4 response engine loaded successfully


In [22]:
# CELL 11A - Unified Professor Stress Test

professor_test_questions = [

    # LEGAL / OWNERSHIP
    "Can a foreigner buy land in Bangkok?",
    "Can a German buy a condominium in Bangkok?",
    "I want a leasehold house in Bangkok. What should I know?",

    # VALUATION
    "What is the difference between gross rental yield and cap rate?",
    "How do I calculate NPV for a real estate investment?",

    # TECHNICAL
    "What should I check regarding flood and soil conditions?",
    "What flood risks should I consider when buying a condo in On Nut?",

    # AREA SCREENING / COMPARISON
    "Is Bang Kapi attractive for rental income?",
    "Compare Bang Kapi with On Nut for rental income.",

    # RECOMMENDATION
    "I have THB 5 million and want good rental income. Which Bangkok areas should I consider?",
    "Which Bangkok areas are best for capital growth?",

    # GENERAL / AMBIGUOUS
    "What should I check before investing in Bangkok real estate?"
]


for i, question in enumerate(professor_test_questions, 1):

    print("\n" + "=" * 100)
    print(f"PROFESSOR TEST {i}")
    print("=" * 100)
    print("QUESTION:")
    print(question)
    print()

    parsed = parse_investment_query(question)
    route = route_investment_query(parsed)

    print(
        "DETECTED ROUTE:",
        route["primary_route"]
    )

    print(
        "LOCATIONS:",
        parsed.get("locations", [])
    )

    print(
        "PROPERTY TYPES:",
        parsed.get("property_types", [])
    )

    print(
        "STRATEGIES:",
        parsed.get("investment_strategies", [])
    )

    print(
        "VALUATION CONCEPTS:",
        parsed.get("valuation_concepts", [])
    )

    print(
        "TECHNICAL RISKS:",
        parsed.get("technical_risks", [])
    )

    print("\nRESPONSE:\n")

    answer = generate_unified_response(
        user_query=question,
        budget_million_thb=8,
        target_size_sqm=40,
        selected_strategy="balanced"
    )

    display(Markdown(answer))


PROFESSOR TEST 1
QUESTION:
Can a foreigner buy land in Bangkok?

DETECTED ROUTE: legal_restriction
LOCATIONS: []
PROPERTY TYPES: ['land']
STRATEGIES: []
VALUATION CONCEPTS: []
TECHNICAL RISKS: []

RESPONSE:



# 🇹🇭 Bangkok Foreign Investor AI

## Regulation-First Legal Screening

**Property type:** Land

**Ownership structure:** Direct ownership / freehold screening

**Regulatory status:** GENERALLY RESTRICTED

**Core rule:** Foreign individuals generally cannot directly own land in Thailand under ordinary circumstances, subject to limited statutory exceptions.

**Key condition:** A foreign investor should not assume ordinary direct freehold land ownership is legally available.

## Required Legal Due Diligence

- Obtain qualified Thai legal advice before committing funds.
- Verify the land title at the competent Land Office.
- Confirm the exact permitted ownership or investment structure.
- Do not use nominee arrangements to circumvent Thai law.
- Review zoning, access, easements and development restrictions.

## Legal Gate

**The chatbot should not proceed directly to an investment ranking for ordinary direct foreign freehold land ownership.**

The legally feasible ownership or investment structure should be clarified first.

Potential alternatives may include professionally reviewed leasehold or other lawful structures, depending on the specific transaction.

## Model Limitation

This is regulation-first investment screening. It does not determine the legal validity of a specific contract, title, ownership structure or transaction.

## Disclaimer

This chatbot is an educational investment-screening prototype and does not provide legal, tax, engineering, valuation or financial advice. Regulations, market conditions and transaction requirements may change. Users should verify current requirements and property-specific information with competent authorities, qualified legal advisers, banks, engineers, valuers and other professional advisers.


PROFESSOR TEST 2
QUESTION:
Can a German buy a condominium in Bangkok?

DETECTED ROUTE: property_screening
LOCATIONS: []
PROPERTY TYPES: ['condominium']
STRATEGIES: []
VALUATION CONCEPTS: []
TECHNICAL RISKS: []

RESPONSE:



# TH Bangkok Foreign Investor AI

## Bangkok Area Investment Screening

No covered Bangkok area could be identified from the question.

Please specify a Bangkok area such as **On Nut, Bang Kapi, Rama 9, Ari or Sathorn**.

## Disclaimer

This chatbot is an educational investment-screening prototype and does not provide legal, tax, engineering, valuation or financial advice. Regulations, market conditions and transaction requirements may change. Users should verify current requirements and property-specific information with competent authorities, qualified legal advisers, banks, engineers, valuers and other professional advisers.


PROFESSOR TEST 3
QUESTION:
I want a leasehold house in Bangkok. What should I know?

DETECTED ROUTE: legal_guidance
LOCATIONS: []
PROPERTY TYPES: ['house']
STRATEGIES: []
VALUATION CONCEPTS: []
TECHNICAL RISKS: []

RESPONSE:



...

# TH Bangkok Foreign Investor AI

## Bangkok Investment Area Screening

**Investor profile used:** THB 8 million budget | 40 sqm target size | Balanced strategy

### Investment Strategy
The screening balances rental yield, growth potential, transit accessibility, international demand and affordability.

## Top Bangkok Areas

### 1. Bang Sue - Tao Poon
**Investment score:** 84.1/100
**Indicative gross rental yield:** 6.25%
**Growth score:** 8/10
**Transit accessibility:** 10/10
**International demand:** 6/10
**Modeled cost for 40 sqm:** THB 4,100,000
**Budget assessment:** Within budget

### 2. On Nut - Phra Khanong
**Investment score:** 83.1/100
**Indicative gross rental yield:** 6.05%
**Growth score:** 8/10
**Transit accessibility:** 9/10
**International demand:** 8/10
**Modeled cost for 40 sqm:** THB 4,600,000
**Budget assessment:** Within budget

### 3. Punnawithi - Udom Suk
**Investment score:** 82.7/100
**Indicative gross rental yield:** 6.05%
**Growth score:** 9/10
**Transit accessibility:** 8/10
**International demand:** 7/10
**Modeled cost for 40 sqm:** THB 4,000,000
**Budget assessment:** Within budget

### 4. Rama 9 - Ratchada
**Investment score:** 80.4/100
**Indicative gross rental yield:** 5.50%
**Growth score:** 9/10
**Transit accessibility:** 9/10
**International demand:** 8/10
**Modeled cost for 40 sqm:** THB 5,600,000
**Budget assessment:** Within budget

### 5. Huai Khwang
**Investment score:** 79.2/100
**Indicative gross rental yield:** 5.75%
**Growth score:** 8/10
**Transit accessibility:** 9/10
**International demand:** 7/10
**Modeled cost for 40 sqm:** THB 4,700,000
**Budget assessment:** Within budget

## Investment Interpretation

Under the selected **balanced** strategy, **Bang Sue - Tao Poon** receives the highest screening score among the evaluated Bangkok zones that best fit the stated profile.

The ranking should be interpreted as an **area-level screening tool**, not as a recommendation to purchase a specific property. Actual investment performance depends on the individual building, unit quality, purchase price, achievable rent, vacancy, operating costs, legal status and transaction terms.

## Budget Fit

17 of the 20 screened Bangkok zones have a modeled 40 sqm unit cost within the stated THB 8 million budget.

Modeled unit cost is calculated from area-level indicative price-per-sqm data and the selected target size. It is not a property valuation or a guarantee that a suitable unit is available at that price.

## Before Investing

- Verify the actual asking price against recent comparable properties.
- Verify achievable rent, vacancy assumptions and recurring ownership costs.
- Confirm foreign ownership eligibility and condominium foreign quota where applicable.
- Review title, legal records, building condition and condominium juristic-person information.
- Conduct property-specific flood, drainage, soil and engineering due diligence where material.
- Consider transaction costs, taxes, financing and exit liquidity before making an investment decision.

## Model Limitation

Area prices, yields, growth, transit and demand indicators are screening-level inputs. They are not transaction-level appraisal evidence. A high area score does not override property-specific legal, technical or financial concerns.

## Disclaimer

This chatbot is an educational investment-screening prototype and does not provide legal, tax, engineering, valuation or financial advice. Regulations, market conditions and transaction requirements may change. Users should verify current requirements and property-specific information with competent authorities, qualified legal advisers, banks, engineers, valuers and other professional advisers.


PROFESSOR TEST 11
QUESTION:
Which Bangkok areas are best for capital growth?

DETECTED ROUTE: area_recommendation
LOCATIONS: []
PROPERTY TYPES: []
STRATEGIES: ['growth']
VALUATION CONCEPTS: []
TECHNICAL RISKS: []

RESPONSE:



# TH Bangkok Foreign Investor AI

## Bangkok Investment Area Screening

**Investor profile used:** THB 8 million budget | 40 sqm target size | Balanced strategy

### Investment Strategy
The screening balances rental yield, growth potential, transit accessibility, international demand and affordability.

## Top Bangkok Areas

### 1. Bang Sue - Tao Poon
**Investment score:** 84.1/100
**Indicative gross rental yield:** 6.25%
**Growth score:** 8/10
**Transit accessibility:** 10/10
**International demand:** 6/10
**Modeled cost for 40 sqm:** THB 4,100,000
**Budget assessment:** Within budget

### 2. On Nut - Phra Khanong
**Investment score:** 83.1/100
**Indicative gross rental yield:** 6.05%
**Growth score:** 8/10
**Transit accessibility:** 9/10
**International demand:** 8/10
**Modeled cost for 40 sqm:** THB 4,600,000
**Budget assessment:** Within budget

### 3. Punnawithi - Udom Suk
**Investment score:** 82.7/100
**Indicative gross rental yield:** 6.05%
**Growth score:** 9/10
**Transit accessibility:** 8/10
**International demand:** 7/10
**Modeled cost for 40 sqm:** THB 4,000,000
**Budget assessment:** Within budget

### 4. Rama 9 - Ratchada
**Investment score:** 80.4/100
**Indicative gross rental yield:** 5.50%
**Growth score:** 9/10
**Transit accessibility:** 9/10
**International demand:** 8/10
**Modeled cost for 40 sqm:** THB 5,600,000
**Budget assessment:** Within budget

### 5. Huai Khwang
**Investment score:** 79.2/100
**Indicative gross rental yield:** 5.75%
**Growth score:** 8/10
**Transit accessibility:** 9/10
**International demand:** 7/10
**Modeled cost for 40 sqm:** THB 4,700,000
**Budget assessment:** Within budget

## Investment Interpretation

Under the selected **balanced** strategy, **Bang Sue - Tao Poon** receives the highest screening score among the evaluated Bangkok zones that best fit the stated profile.

The ranking should be interpreted as an **area-level screening tool**, not as a recommendation to purchase a specific property. Actual investment performance depends on the individual building, unit quality, purchase price, achievable rent, vacancy, operating costs, legal status and transaction terms.

## Budget Fit

17 of the 20 screened Bangkok zones have a modeled 40 sqm unit cost within the stated THB 8 million budget.

Modeled unit cost is calculated from area-level indicative price-per-sqm data and the selected target size. It is not a property valuation or a guarantee that a suitable unit is available at that price.

## Before Investing

- Verify the actual asking price against recent comparable properties.
- Verify achievable rent, vacancy assumptions and recurring ownership costs.
- Confirm foreign ownership eligibility and condominium foreign quota where applicable.
- Review title, legal records, building condition and condominium juristic-person information.
- Conduct property-specific flood, drainage, soil and engineering due diligence where material.
- Consider transaction costs, taxes, financing and exit liquidity before making an investment decision.

## Model Limitation

Area prices, yields, growth, transit and demand indicators are screening-level inputs. They are not transaction-level appraisal evidence. A high area score does not override property-specific legal, technical or financial concerns.

## Disclaimer

This chatbot is an educational investment-screening prototype and does not provide legal, tax, engineering, valuation or financial advice. Regulations, market conditions and transaction requirements may change. Users should verify current requirements and property-specific information with competent authorities, qualified legal advisers, banks, engineers, valuers and other professional advisers.


PROFESSOR TEST 12
QUESTION:
What should I check before investing in Bangkok real estate?

DETECTED ROUTE: general_due_diligence
LOCATIONS: []
PROPERTY TYPES: []
STRATEGIES: []
VALUATION CONCEPTS: []
TECHNICAL RISKS: []

RESPONSE:



# TH Bangkok Foreign Investor AI

## General Real-Estate Due Diligence

A real-estate investment should normally be reviewed across legal, financial, market, technical and exit-risk dimensions.

### Legal

- Verify title and ownership records.
- Confirm the legally permitted ownership structure.
- Check foreign-ownership restrictions where applicable.
- Review contracts, encumbrances, easements and rights affecting the property.
- Confirm required registrations and approvals.

### Financial

- Verify the exact purchase price and transaction costs.
- Estimate realistic achievable rent rather than relying only on advertised rent.
- Check vacancy assumptions.
- Review common-area fees and recurring operating expenses.
- Stress-test income and resale assumptions.

### Market

- Review recent comparable transaction evidence where available.
- Assess competing supply and development pipeline.
- Evaluate tenant demand.
- Assess resale liquidity.
- Compare the property with relevant alternative locations.

### Technical

- Inspect building condition and maintenance history.
- Review structural and foundation information where relevant.
- Check flood history and drainage.
- Review access during heavy rainfall.
- Consider site-specific geotechnical or engineering due diligence where appropriate.

### Building Condo

- Review condominium juristic-person financial statements.
- Check common-area fees and sinking-fund obligations.
- Review maintenance quality and major planned repairs.
- Confirm remaining foreign ownership quota where applicable.
- Assess unit layout, floor, orientation and building-specific rental demand.

### Exit

- Assess likely buyer demand at resale.
- Review expected holding period.
- Consider transaction costs at exit.
- Avoid assuming future capital appreciation is guaranteed.
- Evaluate whether the investment remains acceptable under weaker resale conditions.

## Model Limitation

Area prices, yields, flood, soil and market indicators are screening-level inputs. They are not transaction-level appraisal evidence and should not replace property-specific legal, technical and financial due diligence.

## Disclaimer

This chatbot is an educational investment-screening prototype and does not provide legal, tax, engineering, valuation or financial advice. Regulations, market conditions and transaction requirements may change. Users should verify current requirements and property-specific information with competent authorities, qualified legal advisers, banks, engineers, valuers and other professional advisers.

In [23]:
# CELL 11B - Critical V4 Routing & Profile Fixes


# ============================================================
# FIX 1:
# Foreign-nationality ownership questions must route to legal
# ============================================================

_route_investment_query_original = route_investment_query


def route_investment_query(parsed):

    query = parsed.get(
        "normalized_query",
        ""
    )

    property_types = parsed.get(
        "property_types",
        []
    )

    foreign_context = parsed.get(
        "foreign_investor",
        {}
    ).get(
        "is_foreign_context",
        False
    )

    ownership_question_terms = [
        "can ",
        "allowed",
        "legally",
        "buy",
        "purchase",
        "own",
        "ownership"
    ]

    is_ownership_question = any(
        term in query
        for term in ownership_question_terms
    )

    # Foreign investor + property ownership question
    if (
        foreign_context
        and property_types
        and is_ownership_question
    ):

        if "land" in property_types:

            return {
                "primary_route": "legal_restriction",
                "secondary_routes": [],
                "reason": (
                    "The question asks whether a foreign investor "
                    "may acquire land in Thailand."
                ),
                "confidence": "high",
                "needs_clarification": False
            }

        return {
            "primary_route": "legal_guidance",
            "secondary_routes": [],
            "reason": (
                "The question asks about property ownership "
                "eligibility for a foreign investor."
            ),
            "confidence": "high",
            "needs_clarification": False
        }

    # All other cases continue through the validated router
    return _route_investment_query_original(
        parsed
    )


# ============================================================
# FIX 2:
# Recommendation engine must respect budget, size and strategy
# detected inside the user's actual question
# ============================================================

_build_area_recommendation_response_original = (
    build_area_recommendation_response
)


def build_area_recommendation_response(
    parsed,
    budget_million_thb=8,
    target_size_sqm=40,
    strategy="balanced"
):

    detected_budget = parsed.get(
        "budget_million_thb"
    )

    detected_size = parsed.get(
        "property_size_sqm"
    )

    final_budget = (
        detected_budget
        if detected_budget is not None
        else budget_million_thb
    )

    final_size = (
        detected_size
        if detected_size is not None
        else target_size_sqm
    )

    final_strategy = resolve_investment_strategy(
        parsed=parsed,
        selected_strategy=strategy
    )

    return _build_area_recommendation_response_original(
        parsed=parsed,
        budget_million_thb=final_budget,
        target_size_sqm=final_size,
        strategy=final_strategy
    )


print("✓ Critical V4 fixes loaded successfully")
print("✓ Foreign-investor ownership routing patched")
print("✓ Parsed budget / size / strategy now override UI defaults")

✓ Critical V4 fixes loaded successfully
✓ Foreign-investor ownership routing patched
✓ Parsed budget / size / strategy now override UI defaults


In [24]:
# CELL 11C - Critical Fix Retest

critical_test_questions = [
    "Can a German buy a condominium in Bangkok?",
    "I have THB 5 million and want good rental income. Which Bangkok areas should I consider?",
    "Which Bangkok areas are best for capital growth?"
]


for i, question in enumerate(
    critical_test_questions,
    1
):

    print("\n" + "=" * 100)
    print(f"CRITICAL RETEST {i}")
    print("=" * 100)

    print("QUESTION:")
    print(question)
    print()

    parsed = parse_investment_query(
        question
    )

    route = route_investment_query(
        parsed
    )

    print(
        "DETECTED ROUTE:",
        route["primary_route"]
    )

    print(
        "DETECTED BUDGET:",
        parsed.get(
            "budget_million_thb"
        )
    )

    print(
        "DETECTED STRATEGY:",
        parsed.get(
            "investment_strategies",
            []
        )
    )

    print("\nRESPONSE:\n")

    answer = generate_unified_response(
        user_query=question,
        budget_million_thb=8,
        target_size_sqm=40,
        selected_strategy="balanced"
    )

    display(
        Markdown(answer)
    )


CRITICAL RETEST 1
QUESTION:
Can a German buy a condominium in Bangkok?

DETECTED ROUTE: legal_guidance
DETECTED BUDGET: None
DETECTED STRATEGY: []

RESPONSE:



# 🇹🇭 Bangkok Foreign Investor AI

## Regulation-First Legal Screening

**Property type:** Condominium

**Ownership structure:** Freehold screening assumed

**Regulatory status:** GENERALLY PERMITTED

**Core rule:** Foreign individuals may generally own qualifying condominium units in Thailand in their own name, subject to applicable statutory conditions and the foreign ownership quota.

**Key condition:** Foreign ownership must generally remain within the applicable condominium foreign-ownership quota.

## Required Legal Due Diligence

- Verify that the condominium is legally registered.
- Confirm the remaining foreign ownership quota with the condominium juristic person.
- Verify the unit title and ownership records with the competent Land Office.
- Confirm foreign-fund remittance and transfer-document requirements before completion.
- Review common-area fees, sinking fund obligations and building financial statements.

## Foreign Condominium Ownership

Foreign condominium ownership is generally subject to the statutory foreign-ownership quota. For screening purposes, this is commonly described as a limit of approximately **49% of the total unit area of the condominium** being foreign-owned.

The available foreign quota must therefore be verified for the specific condominium before transfer.

## Model Limitation

This is regulation-first investment screening. It does not determine the legal validity of a specific contract, title, ownership structure or transaction.

## Disclaimer

This chatbot is an educational investment-screening prototype and does not provide legal, tax, engineering, valuation or financial advice. Regulations, market conditions and transaction requirements may change. Users should verify current requirements and property-specific information with competent authorities, qualified legal advisers, banks, engineers, valuers and other professional advisers.


CRITICAL RETEST 2
QUESTION:
I have THB 5 million and want good rental income. Which Bangkok areas should I consider?

DETECTED ROUTE: area_recommendation
DETECTED BUDGET: 5.0
DETECTED STRATEGY: ['income']

RESPONSE:



# TH Bangkok Foreign Investor AI

## Bangkok Investment Area Screening

**Investor profile used:** THB 5 million budget | 40 sqm target size | Income strategy

### Investment Strategy
The screening places greater emphasis on gross rental yield and affordability while retaining market-demand and transit factors.

## Top Bangkok Areas

### 1. Bang Sue - Tao Poon
**Investment score:** 82.5/100
**Indicative gross rental yield:** 6.25%
**Growth score:** 8/10
**Transit accessibility:** 10/10
**International demand:** 6/10
**Modeled cost for 40 sqm:** THB 4,100,000
**Budget assessment:** Within budget

### 2. On Nut - Phra Khanong
**Investment score:** 80.3/100
**Indicative gross rental yield:** 6.05%
**Growth score:** 8/10
**Transit accessibility:** 9/10
**International demand:** 8/10
**Modeled cost for 40 sqm:** THB 4,600,000
**Budget assessment:** Within budget

### 3. Talat Phlu - Wutthakat
**Investment score:** 80.3/100
**Indicative gross rental yield:** 6.35%
**Growth score:** 7/10
**Transit accessibility:** 8/10
**International demand:** 5/10
**Modeled cost for 40 sqm:** THB 3,100,000
**Budget assessment:** Within budget

### 4. Punnawithi - Udom Suk
**Investment score:** 79.5/100
**Indicative gross rental yield:** 6.05%
**Growth score:** 9/10
**Transit accessibility:** 8/10
**International demand:** 7/10
**Modeled cost for 40 sqm:** THB 4,000,000
**Budget assessment:** Within budget

### 5. Ramkhamhaeng - Bang Kapi
**Investment score:** 76.7/100
**Indicative gross rental yield:** 6.15%
**Growth score:** 8/10
**Transit accessibility:** 7/10
**International demand:** 5/10
**Modeled cost for 40 sqm:** THB 3,300,000
**Budget assessment:** Within budget

## Investment Interpretation

Under the selected **income** strategy, **Bang Sue - Tao Poon** receives the highest screening score among the evaluated Bangkok zones that best fit the stated profile.

The ranking should be interpreted as an **area-level screening tool**, not as a recommendation to purchase a specific property. Actual investment performance depends on the individual building, unit quality, purchase price, achievable rent, vacancy, operating costs, legal status and transaction terms.

## Budget Fit

11 of the 20 screened Bangkok zones have a modeled 40 sqm unit cost within the stated THB 5 million budget.

Modeled unit cost is calculated from area-level indicative price-per-sqm data and the selected target size. It is not a property valuation or a guarantee that a suitable unit is available at that price.

## Before Investing

- Verify the actual asking price against recent comparable properties.
- Verify achievable rent, vacancy assumptions and recurring ownership costs.
- Confirm foreign ownership eligibility and condominium foreign quota where applicable.
- Review title, legal records, building condition and condominium juristic-person information.
- Conduct property-specific flood, drainage, soil and engineering due diligence where material.
- Consider transaction costs, taxes, financing and exit liquidity before making an investment decision.

## Model Limitation

Area prices, yields, growth, transit and demand indicators are screening-level inputs. They are not transaction-level appraisal evidence. A high area score does not override property-specific legal, technical or financial concerns.

## Disclaimer

This chatbot is an educational investment-screening prototype and does not provide legal, tax, engineering, valuation or financial advice. Regulations, market conditions and transaction requirements may change. Users should verify current requirements and property-specific information with competent authorities, qualified legal advisers, banks, engineers, valuers and other professional advisers.


CRITICAL RETEST 3
QUESTION:
Which Bangkok areas are best for capital growth?

DETECTED ROUTE: area_recommendation
DETECTED BUDGET: None
DETECTED STRATEGY: ['growth']

RESPONSE:



# TH Bangkok Foreign Investor AI

## Bangkok Investment Area Screening

**Investor profile used:** THB 8 million budget | 40 sqm target size | Growth strategy

### Investment Strategy
The screening places greater emphasis on expected area-level growth potential and transit accessibility.

## Top Bangkok Areas

### 1. Rama 9 - Ratchada
**Investment score:** 83.5/100
**Indicative gross rental yield:** 5.50%
**Growth score:** 9/10
**Transit accessibility:** 9/10
**International demand:** 8/10
**Modeled cost for 40 sqm:** THB 5,600,000
**Budget assessment:** Within budget

### 2. Punnawithi - Udom Suk
**Investment score:** 83.4/100
**Indicative gross rental yield:** 6.05%
**Growth score:** 9/10
**Transit accessibility:** 8/10
**International demand:** 7/10
**Modeled cost for 40 sqm:** THB 4,000,000
**Budget assessment:** Within budget

### 3. Bang Sue - Tao Poon
**Investment score:** 82.5/100
**Indicative gross rental yield:** 6.25%
**Growth score:** 8/10
**Transit accessibility:** 10/10
**International demand:** 6/10
**Modeled cost for 40 sqm:** THB 4,100,000
**Budget assessment:** Within budget

### 4. On Nut - Phra Khanong
**Investment score:** 82.4/100
**Indicative gross rental yield:** 6.05%
**Growth score:** 8/10
**Transit accessibility:** 9/10
**International demand:** 8/10
**Modeled cost for 40 sqm:** THB 4,600,000
**Budget assessment:** Within budget

### 5. Bang Na
**Investment score:** 80.0/100
**Indicative gross rental yield:** 6.00%
**Growth score:** 9/10
**Transit accessibility:** 7/10
**International demand:** 6/10
**Modeled cost for 40 sqm:** THB 3,600,000
**Budget assessment:** Within budget

## Investment Interpretation

Under the selected **growth** strategy, **Rama 9 - Ratchada** receives the highest screening score among the evaluated Bangkok zones that best fit the stated profile.

The ranking should be interpreted as an **area-level screening tool**, not as a recommendation to purchase a specific property. Actual investment performance depends on the individual building, unit quality, purchase price, achievable rent, vacancy, operating costs, legal status and transaction terms.

## Budget Fit

17 of the 20 screened Bangkok zones have a modeled 40 sqm unit cost within the stated THB 8 million budget.

Modeled unit cost is calculated from area-level indicative price-per-sqm data and the selected target size. It is not a property valuation or a guarantee that a suitable unit is available at that price.

## Before Investing

- Verify the actual asking price against recent comparable properties.
- Verify achievable rent, vacancy assumptions and recurring ownership costs.
- Confirm foreign ownership eligibility and condominium foreign quota where applicable.
- Review title, legal records, building condition and condominium juristic-person information.
- Conduct property-specific flood, drainage, soil and engineering due diligence where material.
- Consider transaction costs, taxes, financing and exit liquidity before making an investment decision.

## Model Limitation

Area prices, yields, growth, transit and demand indicators are screening-level inputs. They are not transaction-level appraisal evidence. A high area score does not override property-specific legal, technical or financial concerns.

## Disclaimer

This chatbot is an educational investment-screening prototype and does not provide legal, tax, engineering, valuation or financial advice. Regulations, market conditions and transaction requirements may change. Users should verify current requirements and property-specific information with competent authorities, qualified legal advisers, banks, engineers, valuers and other professional advisers.

In [25]:
# CELL 12 - FINAL GRADIO INTERFACE V4


# ============================================================
# 1. SAFE UI WRAPPER
# ============================================================

def chatbot_interface_v4(
    user_question,
    budget_million_thb,
    target_size_sqm,
    investment_strategy
):
    """
    Safe wrapper between the Gradio interface and the unified V4 engine.
    Ordinary unexpected user inputs should not crash the application.
    """

    try:

        return generate_unified_response(
            user_query=user_question,
            budget_million_thb=budget_million_thb,
            target_size_sqm=target_size_sqm,
            selected_strategy=investment_strategy
        )

    except Exception as error:

        return f"""
# TH Bangkok Foreign Investor AI

## Controlled Fallback

The request could not be processed by the specialized investment modules.

Please try rephrasing the question or provide more specific information such as:

- Property type
- Bangkok area
- Investment objective
- Budget
- Target property size
- Legal / ownership question
- Valuation topic
- Technical risk topic

**Technical note:** The prototype encountered an internal processing exception and returned this controlled fallback instead of terminating the application.

## Disclaimer

{LEGAL_DISCLAIMER}
"""


# ============================================================
# 2. FINAL USER INTERFACE
# ============================================================

with gr.Blocks(
    title="TH Bangkok Foreign Investor AI"
) as demo_v4:

    gr.Markdown(
        """
# 🇹🇭 TH Bangkok Foreign Investor AI

### AI-Assisted Real Estate Investment Decision Support for Foreign Investors

**Regulation-first and data-driven Bangkok investment screening**

This educational prototype can assist with:

- Foreign ownership and legal screening
- Bangkok investment-area recommendations
- Single-area investment screening
- Multi-area comparison
- Budget and property-size screening
- Rental-income, growth and low-risk strategies
- Flood, drainage, soil and engineering due diligence
- NPV, IRR, NOI, Cap Rate and other valuation concepts
- General real-estate investment due diligence

The system combines structured Bangkok market indicators with rule-based
legal, technical and investment decision logic.
"""
    )

    gr.Markdown("---")

    with gr.Row():

        # ----------------------------------------------------
        # LEFT SIDE - USER QUESTION
        # ----------------------------------------------------

        with gr.Column(scale=2):

            user_question = gr.Textbox(
                label="Your Investment Question",
                placeholder=(
                    "Example: I am a German investor with THB 6 million. "
                    "Which Bangkok areas are attractive for rental income?"
                ),
                lines=6
            )

            analyze_button = gr.Button(
                "Analyze Investment",
                variant="primary"
            )


        # ----------------------------------------------------
        # RIGHT SIDE - STRUCTURED INVESTOR PROFILE
        # ----------------------------------------------------

        with gr.Column(scale=1):

            budget_input = gr.Number(
                value=8,
                label="Investment Budget (Million THB)"
            )

            size_input = gr.Number(
                value=40,
                label="Target Property Size (sqm)"
            )

            strategy_input = gr.Dropdown(
                choices=[
                    "balanced",
                    "income",
                    "growth",
                    "low_risk"
                ],
                value="balanced",
                label="Investment Strategy"
            )


    gr.Markdown("---")

    gr.Markdown(
        """
### Example Questions

**Legal & ownership**
- Can a German buy a condominium in Bangkok?
- Can a foreigner buy land in Bangkok?
- I want a leasehold house in Bangkok. What should I know?
- What is a Chanote and how is it different from Nor Sor 3?

**Bangkok investment analysis**
- Is Bang Kapi attractive for rental income?
- Compare Bang Kapi with On Nut for rental income.
- Which Bangkok areas are best for capital growth?
- I have THB 5 million and want good rental income. Which areas should I consider?

**Technical due diligence**
- What should I check regarding flood and soil conditions?
- What flood risks should I consider when buying a condo in On Nut?
- What structural problems should I check before buying an older condominium?

**Valuation**
- What is the difference between gross rental yield and cap rate?
- How do I calculate NPV for a real estate investment?
- What is NOI?
- What is the Income Approach?

**General investment**
- What should I check before investing in Bangkok real estate?
"""
    )

    gr.Markdown("---")

    output = gr.Markdown(
        label="Investment Analysis"
    )


    # ========================================================
    # 3. BUTTON CONNECTION
    # ========================================================

    analyze_button.click(
        fn=chatbot_interface_v4,
        inputs=[
            user_question,
            budget_input,
            size_input,
            strategy_input
        ],
        outputs=output
    )


    # ========================================================
    # 4. ENTER-KEY SUPPORT
    # ========================================================

    user_question.submit(
        fn=chatbot_interface_v4,
        inputs=[
            user_question,
            budget_input,
            size_input,
            strategy_input
        ],
        outputs=output
    )


    # ========================================================
    # 5. FOOTER
    # ========================================================

    gr.Markdown(
        f"""
---

**Prototype scope:** Bangkok foreign-investor real-estate investment screening.

**Method:** Rule-based query understanding + structured legal, technical,
valuation and multi-criteria investment decision modules.

**Important:** {LEGAL_DISCLAIMER}
"""
    )


# ============================================================
# 6. LAUNCH
# ============================================================

demo_v4.launch(
    share=True,
    debug=False,
    show_error=False
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://70238273a2654a8d1e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
